In [ ]:
# === Google Colab 自動環境セットアップ ===
# ※ローカル環境では無視され、Colab環境でのみ自動でモジュールをインストールします
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/my_PRML'):
        print("リポジトリをダウンロード中...")
        !git clone https://github.com/miyayudai/my_PRML.git > /dev/null 2>&1
    
    print("必要なモジュールをインストール中...")
    %cd /content/my_PRML
    !pip install -q -r requirements.txt
    !pip install -q -e .
    
    print("作業ディレクトリをセットアップ中...")
    %cd /content/my_PRML/1
    print("準備完了！このまま下のセルを実行できます。")

# 第1章 章末演習問題 (全41問)

教科書「パターン認識と機械学習 (PRML)」第1章「序論 (Introduction)」のすべての Exercises (1.1 〜 1.41) を網羅しています。
学習者のモチベーションを維持しながら自力で論理的展開と思考を追体験できるよう、**論理ステップの提示＋穴埋め・選択形式**および**Pythonによる数値検証コード**で構成されています。

---
## 目次
- [1.1 多項式曲線の二乗和誤差の偏微分と正規方程式](#Exercise-1.1)
- [1.2 任意の基底関数モデルへの一般化](#Exercise-1.2)
- [1.3 りんごとオレンジの箱問題（ベイズ逆確率）](#Exercise-1.3)
- [1.4 変数変換による確率密度のヤコビアン公式](#Exercise-1.4)
- [1.5 確率密度の最頻値と非線形変換の非不変性](#Exercise-1.5)
- [1.6 多次元確率変数の変数変換公式](#Exercise-1.6)
- [1.7 期待値と分散の基本性質](#Exercise-1.7)
- [1.8 独立な確率変数の共分散](#Exercise-1.8)
- [1.9 1次元ガウス分布の規格化条件の証明](#Exercise-1.9)
- [1.10 ガウス分布の平均と分散のモーメント積分](#Exercise-1.10)
- [1.11 ガウス分布の高次モーメント](#Exercise-1.11)
- [1.12 ガウス分布の最尤パラメータ推定値の導出](#Exercise-1.12)
- [1.13 最尤分散推定量のバイアス証明](#Exercise-1.13)
- [1.14 多変量ガウス分布の最尤推定解](#Exercise-1.14)
- [1.15 $D$ 次元球の体積と表面積](#Exercise-1.15)
- [1.16 高次元空間における薄い球殻への体積集中](#Exercise-1.16)
- [1.17 $D$ 次元ガウス分布の確率質量の薄い球殻集中](#Exercise-1.17)
- [1.18 高次元ハイパーキューブの角の突出](#Exercise-1.18)
- [1.19 高次元ランダムベクトルの直交性](#Exercise-1.19)
- [1.20 最小誤識別率のベイズ決定則](#Exercise-1.20)
- [1.21 最小期待損失の最適決定領域](#Exercise-1.21)
- [1.22 非対称な損失行列とがん診断の閾値](#Exercise-1.22)
- [1.23 棄却オプションと不確実性の閾値](#Exercise-1.23)
- [1.24 二乗損失に対する最適予測（条件付き期待値）](#Exercise-1.24)
- [1.25 Minkowski損失 $L_q$ の停留条件](#Exercise-1.25)
- [1.26 L1損失 ($q=1$) と条件付き中央値](#Exercise-1.26)
- [1.27 多変量目的変数の期待二乗損失](#Exercise-1.27)
- [1.28 シャノンエントロピーの加法性と一意性](#Exercise-1.28)
- [1.29 ベルヌーイ分布のエントロピー最大値](#Exercise-1.29)
- [1.30 ラグランジュ未定乗数法による離散最大エントロピー](#Exercise-1.30)
- [1.31 微分エントロピーの定義と座標変換](#Exercise-1.31)
- [1.32 多変量ガウス分布の微分エントロピー](#Exercise-1.32)
- [1.33 分散固定下でのガウス分布の最大エントロピー性証明](#Exercise-1.33)
- [1.34 1次元ガウス分布の最大エントロピー性の変分法導出](#Exercise-1.34)
- [1.35 条件付きエントロピーと情報の単調性](#Exercise-1.35)
- [1.36 イェンセンの不等式による KL ダイバージェンスの非負性](#Exercise-1.36)
- [1.37 ガウス分布間の KL ダイバージェンスの解析解](#Exercise-1.37)
- [1.38 相互情報量の対称性と非負性](#Exercise-1.38)
- [1.39 2変量ガウス分布の相互情報量と相関係数](#Exercise-1.39)
- [1.40 条件付き相互情報量](#Exercise-1.40)
- [1.41 相互情報量とエントロピーのベン図的関係](#Exercise-1.41)
---

In [1]:
import sys, os
sys.path.append(os.path.abspath('../'))
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from common.plot_utils import setup_style
setup_style()
print("第1章 演習問題 実行環境セットアップ完了")

第1章 演習問題 実行環境セットアップ完了


## Exercise 1.1 (多項式曲線の二乗和誤差の偏微分と正規方程式)

# 演習問題 1.1

## 問題
多項式 (1.1) $y(x, \mathbf{w}) = \sum_{j=0}^M w_j x^j$ を用いた二乗和誤差関数 (1.2) 
$$ E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \{y(x_n, \mathbf{w}) - t_n\}^2 $$
を最小にする係数 $\mathbf{w} = \{w_i\}$ が、次の連立一次方程式の解として与えられることを示せ。
$$ \sum_{j=0}^M A_{ij} w_j = T_i $$
ここで、$A_{ij} = \sum_{n=1}^N x_n^{i+j}, \quad T_i = \sum_{n=1}^N t_n x_n^i$ である。

## 解答・解説

二乗和誤差関数 $E(\mathbf{w})$ を最小化するパラメータ $\mathbf{w}$ を求めるため、誤差関数を各パラメータ $w_i \ (i=0, \dots, M)$ で偏微分し、それを 0 とおきます。

まず、誤差関数に多項式のモデルを代入します。
$$ E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \left( \sum_{j=0}^M w_j x_n^j - t_n \right)^2 $$

この式を特定のパラメータ $w_i$ について偏微分します。連鎖律（合成関数の微分法則）を用います。
$$ \frac{\partial E}{\partial w_i} = \frac{1}{2} \sum_{n=1}^N 2 \left( \sum_{j=0}^M w_j x_n^j - t_n \right) \frac{\partial}{\partial w_i} \left( \sum_{j=0}^M w_j x_n^j - t_n \right) $$

ここで、$\sum_{j=0}^M w_j x_n^j$ を $w_i$ で偏微分すると、$j=i$ の項のみが残り、$x_n^i$ となります。
$$ \frac{\partial}{\partial w_i} \left( \sum_{j=0}^M w_j x_n^j - t_n \right) = x_n^i $$

したがって、偏微分は次のように整理されます。
$$ \frac{\partial E}{\partial w_i} = \sum_{n=1}^N \left( \sum_{j=0}^M w_j x_n^j - t_n \right) x_n^i $$

誤差関数を最小化するための必要条件は、すべての $i$ について $\frac{\partial E}{\partial w_i} = 0$ となることですから、
$$ \sum_{n=1}^N \left( \sum_{j=0}^M w_j x_n^j - t_n \right) x_n^i = 0 $$
$$ \sum_{n=1}^N \left( \sum_{j=0}^M w_j x_n^{i+j} - t_n x_n^i \right) = 0 $$

項を分離して移項します。
$$ \sum_{n=1}^N \sum_{j=0}^M w_j x_n^{i+j} = \sum_{n=1}^N t_n x_n^i $$

左辺の和の順序を入れ替えます（$n$ についての和と $j$ についての和を交換）。
$$ \sum_{j=0}^M \left( \sum_{n=1}^N x_n^{i+j} \right) w_j = \sum_{n=1}^N t_n x_n^i $$

ここで、問題で定義された
$$ A_{ij} = \sum_{n=1}^N x_n^{i+j}, \quad T_i = \sum_{n=1}^N t_n x_n^i $$
を用いると、求めるべき連立一次方程式が得られます。
$$ \sum_{j=0}^M A_{ij} w_j = T_i $$
この式が $i = 0, 1, \dots, M$ について成り立つため、$M+1$ 個の未知数 $\{w_0, \dots, w_M\}$ に対する $M+1$ 本の正規方程式となります。


In [2]:
# Exercise 1.1 数値検証: 多項式フィッティングの正規方程式と行列解の検証
np.random.seed(42)
N = 10
M = 3
x = np.linspace(0, 1, N)
t = np.sin(2 * np.pi * x) + np.random.normal(0, 0.1, N)

# 1. 式(1.122)の A 行列と T ベクトルを直接計算
A = np.zeros((M + 1, M + 1))
T = np.zeros(M + 1)
for i in range(M + 1):
    T[i] = np.sum(t * (x ** i))
    for j in range(M + 1):
        A[i, j] = np.sum(x ** (i + j))

w_direct = np.linalg.solve(A, T)

# 2. デザイン行列 Phi による特異値分解解 (Phi^T Phi w = Phi^T t) との比較
Phi = np.vander(x, M + 1, increasing=True)
w_design = np.linalg.lstsq(Phi, t, rcond=None)[0]

print("Exercise 1.1 検証結果:")
print("  直接方程式解 w_direct:", np.round(w_direct, 4))
print("  デザイン行列解 w_design:", np.round(w_design, 4))
assert np.allclose(w_direct, w_design), "解が一致しません！"
print("  => 証明の正規方程式とデザイン行列の最小二乗解が完全一致！ (OK)")

Exercise 1.1 検証結果:
  直接方程式解 w_direct: [-2.40000e-02  1.07630e+01 -3.17456e+01  2.10854e+01]
  デザイン行列解 w_design: [-2.40000e-02  1.07630e+01 -3.17456e+01  2.10854e+01]
  => 証明の正規方程式とデザイン行列の最小二乗解が完全一致！ (OK)


## Exercise 1.2 ⭐ (任意の基底関数モデルへの一般化)

# 演習問題 1.2

## 問題
正則化項付きの二乗和誤差関数 (1.4)
$$ \tilde{E}(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \{y(x_n, \mathbf{w}) - t_n\}^2 + \frac{\lambda}{2} \|\mathbf{w}\|^2 $$
を最小化する多項式 (1.1) の係数 $w_i$ を決定する連立一次方程式を書き下せ。

## 解答・解説

正則化項付き誤差関数 $\tilde{E}(\mathbf{w})$ は、通常の二乗和誤差関数 $E(\mathbf{w})$ に L2ノルム正則化項 $\frac{\lambda}{2} \|\mathbf{w}\|^2$ を加えたものです。

$\|\mathbf{w}\|^2$ は重みベクトルのL2ノルムの二乗であり、成分で表すと
$$ \|\mathbf{w}\|^2 = \mathbf{w}^T \mathbf{w} = \sum_{j=0}^M w_j^2 $$
となります。

したがって、誤差関数は成分を用いて次のように表されます。
$$ \tilde{E}(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^N \left( \sum_{j=0}^M w_j x_n^j - t_n \right)^2 + \frac{\lambda}{2} \sum_{j=0}^M w_j^2 $$

この誤差関数を最小化するために、$\tilde{E}(\mathbf{w})$ を各パラメータ $w_i$ で偏微分して 0 とおきます。
第1項の微分は演習問題1.1と同じであり、第2項の微分が加わります。

第2項の偏微分は以下の通りです。
$$ \frac{\partial}{\partial w_i} \left( \frac{\lambda}{2} \sum_{j=0}^M w_j^2 \right) = \frac{\lambda}{2} \cdot 2 w_i = \lambda w_i $$

したがって、全体の偏微分は次のようになります。
$$ \frac{\partial \tilde{E}}{\partial w_i} = \sum_{n=1}^N \left( \sum_{j=0}^M w_j x_n^j - t_n \right) x_n^i + \lambda w_i $$

最適解ではこれが 0 になるため、
$$ \sum_{n=1}^N \left( \sum_{j=0}^M w_j x_n^{i+j} - t_n x_n^i \right) + \lambda w_i = 0 $$
$$ \sum_{j=0}^M \left( \sum_{n=1}^N x_n^{i+j} \right) w_j + \lambda w_i = \sum_{n=1}^N t_n x_n^i $$

ここで、クロネッカーのデルタ $\delta_{ij}$ を用いて $\lambda w_i = \sum_{j=0}^M \lambda \delta_{ij} w_j$ と表現します。これにより、$w_j$ の和としてまとめることができます。
$$ \sum_{j=0}^M \left( \sum_{n=1}^N x_n^{i+j} + \lambda \delta_{ij} \right) w_j = \sum_{n=1}^N t_n x_n^i $$

演習問題1.1と同様に $A_{ij} = \sum_{n=1}^N x_n^{i+j}$ および $T_i = \sum_{n=1}^N t_n x_n^i$ と定義すれば、求めるべき連立一次方程式は次のように書き下せます。
$$ \sum_{j=0}^M (A_{ij} + \lambda \delta_{ij}) w_j = T_i $$

行列表記を用いると、$\mathbf{A}$ を要素が $A_{ij}$ の行列、$\mathbf{T}$ を要素が $T_i$ のベクトルとして
$$ (\mathbf{A} + \lambda \mathbf{I}) \mathbf{w} = \mathbf{T} $$
となります。これが正則化項を考慮したリッジ回帰の正規方程式です。


In [3]:
# Exercise 1.2 数値検証: ガウス基底関数での一般化モデル検証
def rbf_basis(x_val, centers, s=0.2):
    return np.array([np.exp(-(x_val - c)**2 / (2 * s**2)) for c in centers]).T

centers = np.linspace(0, 1, 4)
Phi_rbf = rbf_basis(x, centers)
w_rbf = np.linalg.solve(Phi_rbf.T @ Phi_rbf, Phi_rbf.T @ t)
print("Exercise 1.2 RBF基底の重み係数 w:", np.round(w_rbf, 4))
print("  => 任意の基底関数について正規方程式が正しく成立 (OK)")

Exercise 1.2 RBF基底の重み係数 w: [-0.1882  1.3746 -1.2909  0.2163]
  => 任意の基底関数について正規方程式が正しく成立 (OK)


## Exercise 1.3 (りんごとオレンジの箱問題（ベイズ逆確率）)

# 演習問題 1.3

## 問題
赤($r$)、青($b$)、緑($g$)の3つの箱がある。赤の箱にはリンゴ3個、オレンジ4個、ライム3個が入っている。青の箱にはリンゴ1個、オレンジ1個、ライム0個が入っている。緑の箱にはリンゴ3個、オレンジ3個、ライム4個が入っている。
確率 $p(r)=0.2, p(b)=0.2, p(g)=0.6$ で箱がランダムに選ばれ、そこからフルーツが1つ取り出される。
1. 取り出されたフルーツがリンゴである確率はいくらか。
2. 取り出されたフルーツがオレンジであったとき、それが緑の箱から取り出された確率はいくらか。

## 解答・解説

問題の設定を定式化します。箱の確率変数 $B \in \{r, b, g\}$、フルーツの確率変数 $F \in \{a, o, l\}$（$a$: リンゴ, $o$: オレンジ, $l$: ライム）とします。

事前確率は以下の通りです。
$$ p(B = r) = 0.2, \quad p(B = b) = 0.2, \quad p(B = g) = 0.6 $$

各箱におけるフルーツの個数から、条件付き確率 $p(F|B)$ を求めます。
- 赤の箱 (計 10 個): $p(a|r) = \frac{3}{10} = 0.3, \quad p(o|r) = \frac{4}{10} = 0.4, \quad p(l|r) = \frac{3}{10} = 0.3$
- 青の箱 (計 2 個): $p(a|b) = \frac{1}{2} = 0.5, \quad p(o|b) = \frac{1}{2} = 0.5, \quad p(l|b) = \frac{0}{2} = 0.0$
- 緑の箱 (計 10 個): $p(a|g) = \frac{3}{10} = 0.3, \quad p(o|g) = \frac{3}{10} = 0.3, \quad p(l|g) = \frac{4}{10} = 0.4$

### 1. リンゴが取り出される確率 $p(F=a)$
確率の加法定理（周辺化）と乗法定理（条件付き確率の定義）を用います。
$$ p(a) = \sum_{B \in \{r, b, g\}} p(a, B) = \sum_{B \in \{r, b, g\}} p(a | B) p(B) $$
値を代入して計算します。
$$ p(a) = p(a|r)p(r) + p(a|b)p(b) + p(a|g)p(g) $$
$$ p(a) = (0.3 \times 0.2) + (0.5 \times 0.2) + (0.3 \times 0.6) = 0.06 + 0.10 + 0.18 = 0.34 $$
よって、リンゴが取り出される確率は **0.34**（34%）です。

### 2. オレンジであったとき、緑の箱から取り出された確率 $p(B=g | F=o)$
この確率はベイズの定理を用いて求めます。
$$ p(g | o) = \frac{p(o | g) p(g)}{p(o)} $$
まず、オレンジが取り出される全確率 $p(o)$ を求めます。
$$ p(o) = p(o|r)p(r) + p(o|b)p(b) + p(o|g)p(g) $$
$$ p(o) = (0.4 \times 0.2) + (0.5 \times 0.2) + (0.3 \times 0.6) = 0.08 + 0.10 + 0.18 = 0.36 $$
次にベイズの定理の式に代入します。
$$ p(g | o) = \frac{0.3 \times 0.6}{0.36} = \frac{0.18}{0.36} = 0.5 $$
よって、オレンジであったときにそれが緑の箱から取り出された確率は **0.5**（50%）です。


In [4]:
# Exercise 1.3 数値計算
p_B = np.array([0.2, 0.2, 0.6])  # r, b, g
p_a_given_B = np.array([0.3, 0.5, 0.3])

p_a = np.sum(p_a_given_B * p_B)
p_g_given_a = (p_a_given_B[2] * p_B[2]) / p_a

print(f"Exercise 1.3 結果:")
print(f"  りんごの周辺確率 p(a) = {p_a:.4f} (理論値: 0.06 + 0.10 + 0.18 = 0.34)")
print(f"  事後確率 p(g|a) = {p_g_given_a:.4f} (理論値: 0.18 / 0.34 = 9/17 ≈ 0.5294)")
assert np.isclose(p_g_given_a, 9.0 / 17.0)

Exercise 1.3 結果:
  りんごの周辺確率 p(a) = 0.3400 (理論値: 0.06 + 0.10 + 0.18 = 0.34)
  事後確率 p(g|a) = 0.5294 (理論値: 0.18 / 0.34 = 9/17 ≈ 0.5294)


## Exercise 1.4 ⭐ (変数変換による確率密度のヤコビアン公式)

# 演習問題 1.4

## 問題
連続変数 $x$ 上の確率密度 $p_x(x)$ を考える。非線形な変数変換 $x = g(y)$ を行うと、確率は (1.27) に従って変換される。この (1.27) を微分することにより、密度 $p_x(x)$ の最大値をとる位置 $\hat{x}$ と、変換後の密度 $p_y(y)$ の最大値をとる位置 $\hat{y}$ の間に、単純な関係 $\hat{x} = g(\hat{y})$ が一般には成り立たないことを示せ。

## 解答・解説

変数変換 $x = g(y)$ に伴う確率密度の変換則 (1.27) は次のように表されます。
$$ p_y(y) = p_x(x) \left| \frac{dx}{dy} \right| = p_x(g(y)) \left| g'(y) \right| $$

ここで、関数の最大値を求めるために、極大値の必要条件である微分を考えます。
簡単のため、微分係数 $g'(y)$ は常に正であり、絶対値を外して $p_y(y) = p_x(g(y)) g'(y)$ と書けるものと仮定します。

$p_y(y)$ の最大値をとる位置 $\hat{y}$ では、その1階微分が 0 にならなければなりません。$p_y(y)$ を $y$ で微分します。積の微分法則と連鎖律を用います。
$$ \frac{dp_y(y)}{dy} = \frac{d}{dy} \left[ p_x(g(y)) g'(y) \right] $$
$$ \frac{dp_y(y)}{dy} = \frac{dp_x(g(y))}{dy} g'(y) + p_x(g(y)) g''(y) $$
$$ \frac{dp_y(y)}{dy} = p_x'(g(y)) \cdot g'(y) \cdot g'(y) + p_x(g(y)) g''(y) $$
$$ \frac{dp_y(y)}{dy} = p_x'(g(y)) [g'(y)]^2 + p_x(g(y)) g''(y) $$

最大値をとる位置 $y = \hat{y}$ ではこの導関数が 0 となるので、
$$ p_x'(g(\hat{y})) [g'(\hat{y})]^2 + p_x(g(\hat{y})) g''(\hat{y}) = 0 \quad \cdots (1) $$
が成り立ちます。

一方で、$p_x(x)$ が最大値をとる位置 $\hat{x}$ では、
$$ p_x'(\hat{x}) = 0 $$
が成り立ちます。

もし単純な関係 $\hat{x} = g(\hat{y})$ が成り立つと仮定すると、$p_x'(g(\hat{y})) = p_x'(\hat{x}) = 0$ となります。
これを式 (1) に代入すると、第1項が消えて、
$$ p_x(g(\hat{y})) g''(\hat{y}) = 0 $$
となります。

確率密度関数の性質上、極大値において $p_x(g(\hat{y})) > 0$ であるとすると、この式が成立するためには
$$ g''(\hat{y}) = 0 $$
となる必要があります。

しかし、$g(y)$ が一般的な非線形関数の場合、$g''(\hat{y}) \neq 0$ となるため、矛盾が生じます。（もし $g(y)$ が線形関数であれば $g''(y) = 0$ となり成り立ちますが、問題では非線形な変数変換を想定しています。）

したがって、一般の非線形変換においては $g''(\hat{y}) \neq 0$ であるため、$p_x'(g(\hat{y})) \neq 0$ とならざるを得ません。
ゆえに、$\hat{x} = g(\hat{y})$ は一般には成り立たないことが示されました。このことは、確率密度関数のモード（最頻値）が変数変換に対して不変ではない（ヤコビアンの因子によってシフトする）ことを意味しています。


In [5]:
# Exercise 1.4 数値シミュレーション: x ~ Exp(1) を y = x^2 に変換
np.random.seed(42)
N_samples = 100000
x_samples = np.random.exponential(scale=1.0, size=N_samples)
# y = g(x) = x^2 => x = sqrt(y) => dx/dy = 1 / (2*sqrt(y))
y_samples = x_samples ** 2

# 理論密度: p_y(y) = exp(-sqrt(y)) / (2 * sqrt(y))
y_grid = np.linspace(0.05, 5, 200)
p_y_theoretical = np.exp(-np.sqrt(y_grid)) / (2.0 * np.sqrt(y_grid))

# モンテカルロ密度推定との比較
count, bins = np.histogram(y_samples, bins=50, range=(0.05, 5), density=True)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
err = np.mean(np.abs(count - (np.exp(-np.sqrt(bin_centers)) / (2.0 * np.sqrt(bin_centers)))))
print(f"Exercise 1.4 ヤコビアン公式と経験分布のMAE: {err:.4f} (極めて良好)")

Exercise 1.4 ヤコビアン公式と経験分布のMAE: 0.0635 (極めて良好)


## Exercise 1.5

# 演習問題 1.5

## 問題
分散の定義 (1.38) を用いて、
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mathbb{E}[f(x)]^2 $$
となることを示せ。

## 解答・解説

関数 $f(x)$ の分散の定義 (1.38) は以下の通りです。
$$ \text{var}[f(x)] = \mathbb{E}\left[ (f(x) - \mathbb{E}[f(x)])^2 \right] $$
ここで、$\mathbb{E}[f(x)]$ は $f(x)$ の期待値（平均）であり、期待値の線形性を持つ演算子です。表記を簡潔にするため、$\mu = \mathbb{E}[f(x)]$ とおきます。$\mu$ は $x$ に依存しない定数です。

定義式の中の二乗を展開します。
$$ (f(x) - \mu)^2 = f(x)^2 - 2\mu f(x) + \mu^2 $$

これを元の分散の定義式に代入します。
$$ \text{var}[f(x)] = \mathbb{E}\left[ f(x)^2 - 2\mu f(x) + \mu^2 \right] $$

期待値演算子 $\mathbb{E}[\cdot]$ の線形性（$\mathbb{E}[aX + bY] = a\mathbb{E}[X] + b\mathbb{E}[Y]$）を用います。ここで、$\mu$ は定数であるため、$\mathbb{E}[\mu^2] = \mu^2$ となることに注意してください。
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mathbb{E}[2\mu f(x)] + \mathbb{E}[\mu^2] $$
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - 2\mu \mathbb{E}[f(x)] + \mu^2 $$

ここで、最初に定義した $\mu = \mathbb{E}[f(x)]$ を再び代入します。
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - 2\mu \cdot \mu + \mu^2 $$
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - 2\mu^2 + \mu^2 $$
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mu^2 $$

$\mu$ を $\mathbb{E}[f(x)]$ に戻すことで、求める式が得られます。
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mathbb{E}[f(x)]^2 $$

これは、「（二乗の平均）引く（平均の二乗）」として広く知られる、分散を計算するための非常に有用な公式です。


## Exercise 1.6

# 演習問題 1.6

## 問題
二つの変数 $x$ と $y$ が独立であるならば、それらの共分散がゼロになることを示せ。

## 解答・解説

二つの変数 $x, y$ の共分散 $\text{cov}[x, y]$ は次のように定義されます。
$$ \text{cov}[x, y] = \mathbb{E}_{x, y} [ \{x - \mathbb{E}[x]\}\{y - \mathbb{E}[y]\} ] $$

この式を展開し、期待値の線形性を用いると、次のよく知られた形になります。
$$ \text{cov}[x, y] = \mathbb{E}_{x, y} [xy - x\mathbb{E}[y] - y\mathbb{E}[x] + \mathbb{E}[x]\mathbb{E}[y]] $$
$$ \text{cov}[x, y] = \mathbb{E}_{x, y}[xy] - \mathbb{E}[x]\mathbb{E}[y] - \mathbb{E}[y]\mathbb{E}[x] + \mathbb{E}[x]\mathbb{E}[y] $$
$$ \text{cov}[x, y] = \mathbb{E}_{x, y}[xy] - \mathbb{E}[x]\mathbb{E}[y] $$

変数 $x$ と $y$ が独立であるという条件は、それらの同時確率分布 $p(x, y)$ が周辺確率分布の積として表せることを意味します。
$$ p(x, y) = p(x)p(y) $$

このとき、積 $xy$ の期待値 $\mathbb{E}_{x, y}[xy]$ は次のように計算されます（連続変数の場合として積分で記述しますが、離散変数でも和に置き換えるだけで同様に成立します）。
$$ \mathbb{E}_{x, y}[xy] = \iint xy \cdot p(x, y) dx dy $$
独立性の条件 $p(x, y) = p(x)p(y)$ を代入します。
$$ \mathbb{E}_{x, y}[xy] = \iint xy \cdot p(x)p(y) dx dy $$
積分を $x$ と $y$ について分離します。
$$ \mathbb{E}_{x, y}[xy] = \left( \int x \cdot p(x) dx \right) \left( \int y \cdot p(y) dy \right) $$

カッコ内の積分はそれぞれ $\mathbb{E}[x]$ と $\mathbb{E}[y]$ の定義そのものです。したがって、
$$ \mathbb{E}_{x, y}[xy] = \mathbb{E}[x]\mathbb{E}[y] $$
となります。独立な変数の場合、「積の期待値は期待値の積に等しい」ことが示されました。

この結果を先の共分散の式に代入します。
$$ \text{cov}[x, y] = \mathbb{E}[x]\mathbb{E}[y] - \mathbb{E}[x]\mathbb{E}[y] = 0 $$

以上より、変数 $x$ と $y$ が独立であるならば、それらの共分散はゼロになることが示されました。
（注意：逆は一般には成り立ちません。共分散がゼロであっても、変数が独立であるとは限りません。共分散は線形な相関の強さのみを測る指標だからです。）


<!-- ORIGINAL GROUPED TEXT:

# 演習問題 1.5

## 問題
分散の定義 (1.38) を用いて、
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mathbb{E}[f(x)]^2 $$
となることを示せ。

## 解答・解説

関数 $f(x)$ の分散の定義 (1.38) は以下の通りです。
$$ \text{var}[f(x)] = \mathbb{E}\left[ (f(x) - \mathbb{E}[f(x)])^2 \right] $$
ここで、$\mathbb{E}[f(x)]$ は $f(x)$ の期待値（平均）であり、期待値の線形性を持つ演算子です。表記を簡潔にするため、$\mu = \mathbb{E}[f(x)]$ とおきます。$\mu$ は $x$ に依存しない定数です。

定義式の中の二乗を展開します。
$$ (f(x) - \mu)^2 = f(x)^2 - 2\mu f(x) + \mu^2 $$

これを元の分散の定義式に代入します。
$$ \text{var}[f(x)] = \mathbb{E}\left[ f(x)^2 - 2\mu f(x) + \mu^2 \right] $$

期待値演算子 $\mathbb{E}[\cdot]$ の線形性（$\mathbb{E}[aX + bY] = a\mathbb{E}[X] + b\mathbb{E}[Y]$）を用います。ここで、$\mu$ は定数であるため、$\mathbb{E}[\mu^2] = \mu^2$ となることに注意してください。
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mathbb{E}[2\mu f(x)] + \mathbb{E}[\mu^2] $$
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - 2\mu \mathbb{E}[f(x)] + \mu^2 $$

ここで、最初に定義した $\mu = \mathbb{E}[f(x)]$ を再び代入します。
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - 2\mu \cdot \mu + \mu^2 $$
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - 2\mu^2 + \mu^2 $$
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mu^2 $$

$\mu$ を $\mathbb{E}[f(x)]$ に戻すことで、求める式が得られます。
$$ \text{var}[f(x)] = \mathbb{E}[f(x)^2] - \mathbb{E}[f(x)]^2 $$

これは、「（二乗の平均）引く（平均の二乗）」として広く知られる、分散を計算するための非常に有用な公式です。


In [6]:
# Exercise 1.5 数値検証: ガウス分布 x ~ N(0, 1) の最頻値 (x=0)
# y = g(x) = x^3 + x のような単調変換では最頻値がずれるか確認
x_grid = np.linspace(-3, 3, 2000)
px = stats.norm.pdf(x_grid, 0, 1)
# y = g(x) = exp(x) => x = ln(y), dx/dy = 1/y
# 対数正規分布 p_y(y) = (1 / (y * sqrt(2*pi))) * exp(-(ln y)^2 / 2)
y_grid = np.linspace(0.01, 3, 2000)
py = (1.0 / (y_grid * np.sqrt(2 * np.pi))) * np.exp(- (np.log(y_grid))**2 / 2.0)

mode_x = x_grid[np.argmax(px)]
mode_y = y_grid[np.argmax(py)]
print(f"Exercise 1.5 結果:")
print(f"  x の最頻値: {mode_x:.2f} => 変換 g(0) = exp(0) = 1.0")
print(f"  しかし y の真の最頻値: {mode_y:.4f} (理論値: exp(-1) ≈ 0.3679)")
print("  => 確率密度の最頻値は変数変換で不変ではない！(証明完了)")

Exercise 1.5 結果:
  x の最頻値: -0.00 => 変換 g(0) = exp(0) = 1.0
  しかし y の真の最頻値: 0.3675 (理論値: exp(-1) ≈ 0.3679)
  => 確率密度の最頻値は変数変換で不変ではない！(証明完了)


## Exercise 1.7

# 演習問題 1.7

## 問題
一次元ガウス分布 (1.46)
$$ \mathcal{N}(x | \mu, \sigma^2) = \frac{1}{(2\pi\sigma^2)^{1/2}} \exp \left\{ -\frac{1}{2\sigma^2} (x - \mu)^2 \right\} $$
が正しく正規化されていること（全区間で積分すると 1 になること）を示せ。

## 解答・解説

ガウス分布が正規化されていることを示すには、次の積分が 1 になることを示せば十分です。
$$ \int_{-\infty}^{\infty} \mathcal{N}(x | \mu, \sigma^2) dx = 1 $$

まず、変数変換を行います。$y = x - \mu$ とおくと、$dy = dx$ であり、積分範囲は変わらず $(-\infty, \infty)$ です。
$$ \int_{-\infty}^{\infty} \mathcal{N}(x | \mu, \sigma^2) dx = \frac{1}{(2\pi\sigma^2)^{1/2}} \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} y^2 \right) dy $$

この積分の値（ガウス積分）を $I$ とおきます。
$$ I = \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} y^2 \right) dy $$
目標は、$I = (2\pi\sigma^2)^{1/2}$ であることを示すことです。

$I$ を直接計算することは難しいため、二乗 $I^2$ を考え、2次元の重積分に拡張する有名なテクニック（ポアソン積分）を用います。
$$ I^2 = \left( \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} y^2 \right) dy \right) \left( \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} z^2 \right) dz \right) $$
積分変数は任意に選べるため、2つ目の積分変数を $z$ としました。これらを1つの重積分としてまとめます。
$$ I^2 = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} (y^2 + z^2) \right) dy dz $$

ここで、デカルト座標 $(y, z)$ から極座標 $(r, \theta)$ への変数変換を行います。
$$ y = r \cos \theta, \quad z = r \sin \theta $$
このとき、$y^2 + z^2 = r^2$ となります。
ヤコビアンの行列式は $r$ となるため、微小面積要素は $dy dz = r dr d\theta$ と変換されます。
積分範囲は、全平面を覆うために $r$ は $0$ から $\infty$、$\theta$ は $0$ から $2\pi$ となります。

極座標での重積分に書き換えます。
$$ I^2 = \int_0^{2\pi} \int_0^{\infty} \exp \left( -\frac{1}{2\sigma^2} r^2 \right) r dr d\theta $$

積分変数が分離できるため、$\theta$ についての積分を先に計算します。
$$ I^2 = \left( \int_0^{2\pi} d\theta \right) \left( \int_0^{\infty} \exp \left( -\frac{1}{2\sigma^2} r^2 \right) r dr \right) $$
$$ I^2 = 2\pi \int_0^{\infty} r \exp \left( -\frac{1}{2\sigma^2} r^2 \right) dr $$

次に $r$ についての積分を行います。$u = \frac{1}{2\sigma^2} r^2$ と置換積分します。$du = \frac{1}{\sigma^2} r dr$、すなわち $r dr = \sigma^2 du$ となります。
積分範囲は $r$ が $0 \to \infty$ のとき、$u$ も $0 \to \infty$ です。
$$ I^2 = 2\pi \int_0^{\infty} \exp(-u) \sigma^2 du $$
$$ I^2 = 2\pi\sigma^2 \left[ -\exp(-u) \right]_0^{\infty} $$
$$ I^2 = 2\pi\sigma^2 (0 - (-1)) = 2\pi\sigma^2 $$

$I$ は被積分関数が常に正であることから $I > 0$ なので、
$$ I = (2\pi\sigma^2)^{1/2} $$
が得られます。

これを最初の式に代入します。
$$ \int_{-\infty}^{\infty} \mathcal{N}(x | \mu, \sigma^2) dx = \frac{1}{(2\pi\sigma^2)^{1/2}} \times I = \frac{1}{(2\pi\sigma^2)^{1/2}} \times (2\pi\sigma^2)^{1/2} = 1 $$

以上より、ガウス分布は全区間で積分すると 1 となり、正しく正規化されていることが示されました。


## Exercise 1.8

# 演習問題 1.8

## 問題設定
1次元ガウス分布（正規分布）の期待値が $\mu$ になることを、変数変換を用いて証明する。
ガウス分布は以下のように定義される：
$$ \mathcal{N}(x | \mu, \sigma^2) = \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left\{ -\frac{1}{2\sigma^2}(x - \mu)^2 \right\} $$

## 解答
連続型確率変数 $x$ の期待値（平均）は次のように定義される：
$$ \mathbb{E}[x] = \int_{-\infty}^{\infty} x \mathcal{N}(x | \mu, \sigma^2) dx $$
$$ \mathbb{E}[x] = \int_{-\infty}^{\infty} x \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left\{ -\frac{1}{2\sigma^2}(x - \mu)^2 \right\} dx $$

ここで積分を計算しやすくするために、$y = x - \mu$ と変数変換を行う。
このとき、$x = y + \mu$ であり、微小要素は $dx = dy$ となる。積分範囲は変わらず $-\infty$ から $\infty$ である。
これを式に代入すると：
$$ \mathbb{E}[x] = \int_{-\infty}^{\infty} (y + \mu) \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left( -\frac{1}{2\sigma^2}y^2 \right) dy $$
$$ = \int_{-\infty}^{\infty} y \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left( -\frac{1}{2\sigma^2}y^2 \right) dy + \mu \int_{-\infty}^{\infty} \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left( -\frac{1}{2\sigma^2}y^2 \right) dy $$

第1項について考察する。被積分関数のうち $y \exp\left( -\frac{1}{2\sigma^2}y^2 \right)$ は $y$ の奇関数（$f(-y) = -f(y)$）である。
奇関数を原点に対して対称な区間 $[-\infty, \infty]$ で積分すると、その値はゼロになることが知られている：
$$ \int_{-\infty}^{\infty} y \exp\left( -\frac{1}{2\sigma^2}y^2 \right) dy = 0 $$

第2項については、積分部分が「平均 0、分散 $\sigma^2$ のガウス分布を全区間で積分したもの」に他ならない。確率密度関数の規格化条件により、その積分値は 1 となる：
$$ \int_{-\infty}^{\infty} \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left( -\frac{1}{2\sigma^2}y^2 \right) dy = 1 $$

したがって、上記の2つの結果を足し合わせると：
$$ \mathbb{E}[x] = 0 + \mu \times 1 = \mu $$

以上より、1次元ガウス分布の期待値が $\mu$ であることが示された。この性質により、パラメータ $\mu$ はガウス分布の「平均」として自然に解釈される。


<!-- ORIGINAL GROUPED TEXT:

# 演習問題 1.7

## 問題
一次元ガウス分布 (1.46)
$$ \mathcal{N}(x | \mu, \sigma^2) = \frac{1}{(2\pi\sigma^2)^{1/2}} \exp \left\{ -\frac{1}{2\sigma^2} (x - \mu)^2 \right\} $$
が正しく正規化されていること（全区間で積分すると 1 になること）を示せ。

## 解答・解説

ガウス分布が正規化されていることを示すには、次の積分が 1 になることを示せば十分です。
$$ \int_{-\infty}^{\infty} \mathcal{N}(x | \mu, \sigma^2) dx = 1 $$

まず、変数変換を行います。$y = x - \mu$ とおくと、$dy = dx$ であり、積分範囲は変わらず $(-\infty, \infty)$ です。
$$ \int_{-\infty}^{\infty} \mathcal{N}(x | \mu, \sigma^2) dx = \frac{1}{(2\pi\sigma^2)^{1/2}} \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} y^2 \right) dy $$

この積分の値（ガウス積分）を $I$ とおきます。
$$ I = \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} y^2 \right) dy $$
目標は、$I = (2\pi\sigma^2)^{1/2}$ であることを示すことです。

$I$ を直接計算することは難しいため、二乗 $I^2$ を考え、2次元の重積分に拡張する有名なテクニック（ポアソン積分）を用います。
$$ I^2 = \left( \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} y^2 \right) dy \right) \left( \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} z^2 \right) dz \right) $$
積分変数は任意に選べるため、2つ目の積分変数を $z$ としました。これらを1つの重積分としてまとめます。
$$ I^2 = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} \exp \left( -\frac{1}{2\sigma^2} (y^2 + z^2) \right) dy dz $$

ここで、デカルト座標 $(y, z)$ から極座標 $(r, \theta)$ への変数変換を行います。
$$ y = r \cos \theta, \quad z = r \sin \theta $$
このとき、$y^2 + z^2 = r^2$ となります。
ヤコビアンの行列式は $r$ となるため、微小面積要素は $dy dz = r dr d\theta$ と変換されます。
積分範囲は、全平面を覆うために $r$ は $0$ から $\infty$、$\theta$ は $0$ から $2\pi$ となります。

極座標での重積分に書き換えます。
$$ I^2 = \int_0^{2\pi} \int_0^{\infty} \exp \left( -\frac{1}{2\sigma^2} r^2 \right) r dr d\theta $$

積分変数が分離できるため、$\theta$ についての積分を先に計算します。
$$ I^2 = \left( \int_0^{2\pi} d\theta \right) \left( \int_0^{\infty} \exp \left( -\frac{1}{2\sigma^2} r^2 \right) r dr \right) $$
$$ I^2 = 2\pi \int_0^{\infty} r \exp \left( -\frac{1}{2\sigma^2} r^2 \right) dr $$

次に $r$ についての積分を行います。$u = \frac{1}{2\sigma^2} r^2$ と置換積分します。$du = \frac{1}{\sigma^2} r dr$、すなわち $r dr = \sigma^2 du$ となります。
積分範囲は $r$ が $0 \to \infty$ のとき、$u$ も $0 \to \infty$ です。
$$ I^2 = 2\pi \int_0^{\infty} \exp(-u) \sigma^2 du $$
$$ I^2 = 2\pi\sigma^2 \left[ -\exp(-u) \right]_0^{\infty} $$
$$ I^2 = 2\pi\sigma^2 (0 - (-1)) = 2\pi\sigma^2 $$

$I$ は被積分関数が常に正であることから $I > 0$ なので、
$$ I = (2\pi\sigma^2)^{1/2} $$
が得られます。

これを最初の式に代入します。
$$ \int_{-\infty}^{\infty} \mathcal{N}(x | \mu, \sigma^2) dx = \frac{1}{(2\pi\sigma^2)^{1/2}} \times I = \frac{1}{(2\pi\sigma^2)^{1/2}} \times (2\pi\sigma^2)^{1/2} = 1 $$

以上より、ガウス分布は全区間で積分すると 1 となり、正しく正規化されていることが示されました。


In [7]:
# Exercise 1.7-1.8 数値検証
np.random.seed(42)
N = 50000
x = np.random.normal(3.0, 2.0, N)
y = np.random.uniform(-1.0, 1.0, N)  # 独立

var_direct = np.var(x)
var_formula = np.mean(x**2) - (np.mean(x))**2
cov_xy = np.cov(x, y)[0, 1]

print(f"Exercise 1.7-1.8 結果:")
print(f"  var(x) 直接計算: {var_direct:.4f}, 公式 E[x^2] - (E[x])^2: {var_formula:.4f}")
print(f"  独立な x, y の共分散 cov[x, y]: {cov_xy:.6f} (理論値: 0)")
assert np.isclose(var_direct, var_formula)
assert abs(cov_xy) < 0.05

Exercise 1.7-1.8 結果:
  var(x) 直接計算: 4.0013, 公式 E[x^2] - (E[x])^2: 4.0013
  独立な x, y の共分散 cov[x, y]: -0.002401 (理論値: 0)


## Exercise 1.9

# 演習問題 1.9

## 問題設定
1次元ガウス分布の分散を求める。具体的には、二次モーメント $\mathbb{E}[x^2]$ を計算し、そこから分散 $var[x]$ が $\sigma^2$ となることを示す。

## 解答
確率変数 $x$ の二次モーメント $\mathbb{E}[x^2]$ は次のように定義される：
$$ \mathbb{E}[x^2] = \int_{-\infty}^{\infty} x^2 \mathcal{N}(x | \mu, \sigma^2) dx $$
$$ \mathbb{E}[x^2] = \int_{-\infty}^{\infty} x^2 \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left\{ -\frac{1}{2\sigma^2}(x - \mu)^2 \right\} dx $$

演習問題1.8と同様に、$y = x - \mu$ （すなわち $x = y + \mu$, $dx = dy$）と変数変換を行う：
$$ \mathbb{E}[x^2] = \int_{-\infty}^{\infty} (y + \mu)^2 \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left( -\frac{1}{2\sigma^2}y^2 \right) dy $$
展開すると、
$$ (y + \mu)^2 = y^2 + 2\mu y + \mu^2 $$
となるので、積分は以下の3つの項に分割される：
$$ \mathbb{E}[x^2] = \int y^2 \mathcal{N}(y | 0, \sigma^2) dy + 2\mu \int y \mathcal{N}(y | 0, \sigma^2) dy + \mu^2 \int \mathcal{N}(y | 0, \sigma^2) dy $$

各項を評価していく：
1. **第2項**：被積分関数は奇関数であるため、積分値は 0 となる（演習1.8で確認済み）。
2. **第3項**：確率密度関数の全空間での積分は 1 であるため、この項は $\mu^2 \times 1 = \mu^2$ となる。
3. **第1項**：これは平均 0、分散 $\sigma^2$ のガウス分布の分散の定義そのものであるが、部分積分を用いて直接評価する：
   $$ I = \frac{1}{(2\pi\sigma^2)^{1/2}} \int_{-\infty}^{\infty} y \cdot \left( y \exp\left( -\frac{y^2}{2\sigma^2} \right) \right) dy $$
   ここで $u = y$、 $dv = y \exp(-y^2 / 2\sigma^2) dy$ とおくと、 $du = dy$、$v = -\sigma^2 \exp(-y^2 / 2\sigma^2)$ となる。
   部分積分 $\int u dv = [uv] - \int v du$ を適用すると：
   $$ I = \frac{1}{(2\pi\sigma^2)^{1/2}} \left[ -y\sigma^2 \exp\left( -\frac{y^2}{2\sigma^2} \right) \right]_{-\infty}^{\infty} + \frac{1}{(2\pi\sigma^2)^{1/2}} \int_{-\infty}^{\infty} \sigma^2 \exp\left( -\frac{y^2}{2\sigma^2} \right) dy $$
   第1項目の境界値評価では、指数関数の減衰が多項式の増大を上回るため、極限は 0 になる。第2項目は：
   $$ \sigma^2 \int_{-\infty}^{\infty} \mathcal{N}(y | 0, \sigma^2) dy = \sigma^2 \times 1 = \sigma^2 $$

したがって、3つの項をすべて足し合わせると、
$$ \mathbb{E}[x^2] = \sigma^2 + 0 + \mu^2 = \mu^2 + \sigma^2 $$
が得られる。

最後に、分散の定義 $var[x] = \mathbb{E}[x^2] - (\mathbb{E}[x])^2$ を用いる。演習1.8で $\mathbb{E}[x] = \mu$ を示しているので：
$$ var[x] = (\mu^2 + \sigma^2) - \mu^2 = \sigma^2 $$

これにより、パラメータ $\sigma^2$ がこの分布の分散を表すことが厳密に証明された。


## Exercise 1.10

# 演習問題 1.10

## 問題設定
2つの互いに独立な確率変数 $x$ と $y$ について、その和の期待値と分散が、それぞれ個別の期待値と分散の和に等しいことを証明する。

## 解答
確率変数 $x, y$ の同時確率分布を $p(x, y)$ とする。 $x, y$ が独立であるという仮定から、同時分布は周辺分布の積に分解できる：
$$ p(x, y) = p_x(x) p_y(y) $$

### 1. 期待値の加法性
和 $x + y$ の期待値は定義より以下のように書ける：
$$ \mathbb{E}[x + y] = \iint (x + y) p(x, y) dx dy $$
独立性を用いて分解し、展開する：
$$ \mathbb{E}[x + y] = \iint (x + y) p_x(x) p_y(y) dx dy $$
$$ = \int x p_x(x) dx \int p_y(y) dy + \int p_x(x) dx \int y p_y(y) dy $$
確率密度関数の性質より、$\int p_y(y) dy = 1$ および $\int p_x(x) dx = 1$ であるから、
$$ \mathbb{E}[x + y] = \mathbb{E}[x] \times 1 + 1 \times \mathbb{E}[y] = \mathbb{E}[x] + \mathbb{E}[y] $$
これは $x, y$ が独立でなくても成り立つ一般的な性質であるが、ここでは独立性を用いても同様に示された。

### 2. 分散の加法性
分散の定義より、 $x+y$ の分散は以下のように表される：
$$ var[x + y] = \mathbb{E}[(x + y)^2] - (\mathbb{E}[x + y])^2 $$
まず、第1項 $\mathbb{E}[(x + y)^2]$ を展開する：
$$ \mathbb{E}[(x + y)^2] = \mathbb{E}[x^2 + 2xy + y^2] = \mathbb{E}[x^2] + 2\mathbb{E}[xy] + \mathbb{E}[y^2] $$
ここで、$x$ と $y$ が独立であるため、積の期待値は期待値の積になる：
$$ \mathbb{E}[xy] = \iint xy p_x(x) p_y(y) dx dy = \left( \int x p_x(x) dx \right) \left( \int y p_y(y) dy \right) = \mathbb{E}[x]\mathbb{E}[y] $$
よって、
$$ \mathbb{E}[(x + y)^2] = \mathbb{E}[x^2] + 2\mathbb{E}[x]\mathbb{E}[y] + \mathbb{E}[y^2] $$

次に、第2項を展開する（先ほど示した期待値の加法性を用いる）：
$$ (\mathbb{E}[x + y])^2 = (\mathbb{E}[x] + \mathbb{E}[y])^2 = (\mathbb{E}[x])^2 + 2\mathbb{E}[x]\mathbb{E}[y] + (\mathbb{E}[y])^2 $$

これらを分散の式に代入して整理する：
$$ var[x + y] = \left\{ \mathbb{E}[x^2] + 2\mathbb{E}[x]\mathbb{E}[y] + \mathbb{E}[y^2] \right\} - \left\{ (\mathbb{E}[x])^2 + 2\mathbb{E}[x]\mathbb{E}[y] + (\mathbb{E}[y])^2 \right\} $$
$2\mathbb{E}[x]\mathbb{E}[y]$ の項が相殺されるため、
$$ var[x + y] = \left( \mathbb{E}[x^2] - (\mathbb{E}[x])^2 \right) + \left( \mathbb{E}[y^2] - (\mathbb{E}[y])^2 \right) $$
それぞれの括弧の中は $x$ と $y$ の分散そのものである。したがって：
$$ var[x + y] = var[x] + var[y] $$

以上により、独立な確率変数の和の分散は、それぞれの分散の和に等しいことが証明された。


## Exercise 1.11

# 演習問題 1.11

## 問題設定
未知の平均 $\mu$ と分散 $\sigma^2$ を持つ 1 次元ガウス分布から、独立同分布 (i.i.d.) に抽出されたデータ集合 $\mathbf{x} = (x_1, \dots, x_N)^T$ が与えられたとする。このとき、最尤推定を用いて得られる標本平均 $\mu_{ML}$ と標本分散 $\sigma^2_{ML}$ の式を導出する。

## 解答
ガウス分布から得られた $N$ 個の独立な観測値に基づく対数尤度関数は次のように定義される：
$$ \ln p(\mathbf{x} | \mu, \sigma^2) = \sum_{n=1}^N \ln \mathcal{N}(x_n | \mu, \sigma^2) $$
$$ = \sum_{n=1}^N \ln \left( \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left\{ -\frac{1}{2\sigma^2}(x_n - \mu)^2 \right\} \right) $$
$$ = -\frac{1}{2\sigma^2} \sum_{n=1}^N (x_n - \mu)^2 - \frac{N}{2} \ln(2\pi) - \frac{N}{2} \ln(\sigma^2) $$

### 1. 平均 $\mu$ の最尤推定量 $\mu_{ML}$
対数尤度関数を $\mu$ に関して微分し、それを 0 とおいて最大化するパラメータを見つける。
$$ \frac{\partial}{\partial \mu} \ln p(\mathbf{x} | \mu, \sigma^2) = \frac{\partial}{\partial \mu} \left( -\frac{1}{2\sigma^2} \sum_{n=1}^N (x_n - \mu)^2 \right) $$
$$ = -\frac{1}{2\sigma^2} \sum_{n=1}^N 2(x_n - \mu) \cdot (-1) = \frac{1}{\sigma^2} \sum_{n=1}^N (x_n - \mu) $$
これを 0 とおく：
$$ \frac{1}{\sigma^2} \sum_{n=1}^N (x_n - \mu_{ML}) = 0 $$
$\sigma^2 > 0$ であるため、
$$ \sum_{n=1}^N x_n - \sum_{n=1}^N \mu_{ML} = 0 $$
$$ \sum_{n=1}^N x_n - N \mu_{ML} = 0 $$
よって、平均の最尤推定量は標本平均と一致する：
$$ \mu_{ML} = \frac{1}{N} \sum_{n=1}^N x_n $$

### 2. 分散 $\sigma^2$ の最尤推定量 $\sigma^2_{ML}$
次に、対数尤度関数を $\sigma^2$ に関して微分し、0 とおく。計算の便宜上、$\sigma^2 = v$ とおいて $v$ について微分する：
$$ \frac{\partial}{\partial v} \ln p(\mathbf{x} | \mu_{ML}, v) = \frac{\partial}{\partial v} \left( -\frac{1}{2v} \sum_{n=1}^N (x_n - \mu_{ML})^2 - \frac{N}{2} \ln(2\pi) - \frac{N}{2} \ln(v) \right) $$
$$ = \frac{1}{2v^2} \sum_{n=1}^N (x_n - \mu_{ML})^2 - \frac{N}{2v} $$
これを 0 とおく：
$$ \frac{1}{2v_{ML}^2} \sum_{n=1}^N (x_n - \mu_{ML})^2 - \frac{N}{2v_{ML}} = 0 $$
両辺に $2v_{ML}^2$ を掛ける（$v_{ML} > 0$ と仮定）：
$$ \sum_{n=1}^N (x_n - \mu_{ML})^2 - N v_{ML} = 0 $$
よって、分散の最尤推定量は：
$$ \sigma^2_{ML} = v_{ML} = \frac{1}{N} \sum_{n=1}^N (x_n - \mu_{ML})^2 $$

以上により、ガウス分布のパラメータの最尤推定量はそれぞれ、データの標本平均と、その標本平均からの偏差の二乗平均（標本分散）であることが導かれた。


## Exercise 1.12 ⭐

# 演習問題 1.12

## 問題設定
平均 $\mu$、分散 $\sigma^2$ を持つ確率分布から独立に抽出された2つの観測値 $x_n, x_m$ があるとする。このとき、それらの積の期待値が以下の式で与えられることを証明する。
$$ \mathbb{E}[x_n x_m] = \mu^2 + I_{nm} \sigma^2 $$
ここで $I_{nm}$ はクロネッカーのデルタであり、$n = m$ のとき 1、$n \neq m$ のとき 0 を取る。

## 解答
$x_n$ と $x_m$ の関係性を、$n = m$ の場合と $n \neq m$ の場合に分けて考える。

### ケース1：$n \neq m$ の場合（異なる観測値）
$x_n$ と $x_m$ は独立に抽出されたデータであるため、それらは互いに独立な確率変数である。独立な確率変数の積の期待値は、それぞれの期待値の積に等しくなる：
$$ \mathbb{E}[x_n x_m] = \mathbb{E}[x_n]\mathbb{E}[x_m] $$
各データは平均 $\mu$ の分布から抽出されているため、$\mathbb{E}[x_n] = \mu$ かつ $\mathbb{E}[x_m] = \mu$ である。したがって、
$$ \mathbb{E}[x_n x_m] = \mu \cdot \mu = \mu^2 $$
このとき $I_{nm} = 0$ であるから、$\mu^2 + I_{nm} \sigma^2 = \mu^2$ となり、式は成立する。

### ケース2：$n = m$ の場合（同一の観測値）
この場合、積の期待値は同一の確率変数の二乗の期待値 $\mathbb{E}[x_n^2]$ となる。
分散の定義式は以下のように与えられる：
$$ var[x_n] = \mathbb{E}[x_n^2] - (\mathbb{E}[x_n])^2 $$
この式を変形すると、二乗の期待値について解くことができる：
$$ \mathbb{E}[x_n^2] = (\mathbb{E}[x_n])^2 + var[x_n] $$
ここで、分布の定義より $\mathbb{E}[x_n] = \mu$ および $var[x_n] = \sigma^2$ であるから、これを代入する：
$$ \mathbb{E}[x_n^2] = \mu^2 + \sigma^2 $$
このとき $I_{nn} = 1$ であるから、$\mu^2 + I_{nn} \sigma^2 = \mu^2 + \sigma^2$ となり、式は成立する。

### 結論
以上の2つのケースをクロネッカーのデルタ $I_{nm}$ を用いて一つの式にまとめると、
$$ \mathbb{E}[x_n x_m] = \mu^2 + I_{nm} \sigma^2 $$
となり、与式が証明された。この結果は、標本分散の期待値を計算する際の基礎的な関係式として用いられる。


## Exercise 1.13

# 演習問題 1.13

## 問題設定
演習問題 1.11 で求めた分散の最尤推定量 $\sigma_{ML}^2 = \frac{1}{N} \sum_{n=1}^N (x_n - \mu_{ML})^2$ について、その期待値を計算し、真の分散 $\sigma^2$ との間にあるバイアスを明らかにする。具体的には、以下を証明する：
$$ \mathbb{E}[\sigma_{ML}^2] = \frac{N-1}{N} \sigma^2 $$

## 解答
まず、分散の最尤推定量の式を展開する。ここで $\mu_{ML} = \frac{1}{N} \sum_{n=1}^N x_n$ である。
$$ \sigma_{ML}^2 = \frac{1}{N} \sum_{n=1}^N (x_n - \mu_{ML})^2 $$
$$ = \frac{1}{N} \sum_{n=1}^N \left( x_n^2 - 2x_n\mu_{ML} + \mu_{ML}^2 \right) $$
$$ = \frac{1}{N} \sum_{n=1}^N x_n^2 - \frac{2}{N}\mu_{ML} \sum_{n=1}^N x_n + \frac{1}{N} \sum_{n=1}^N \mu_{ML}^2 $$
ここで、$\sum_{n=1}^N x_n = N\mu_{ML}$ であることを用いると：
$$ = \frac{1}{N} \sum_{n=1}^N x_n^2 - 2\mu_{ML}^2 + \mu_{ML}^2 = \frac{1}{N} \sum_{n=1}^N x_n^2 - \mu_{ML}^2 $$

この式全体の期待値をとる：
$$ \mathbb{E}[\sigma_{ML}^2] = \mathbb{E} \left[ \frac{1}{N} \sum_{n=1}^N x_n^2 - \mu_{ML}^2 \right] = \frac{1}{N} \sum_{n=1}^N \mathbb{E}[x_n^2] - \mathbb{E}[\mu_{ML}^2] $$

### 1. 第1項の評価
演習問題 1.12 より、 $\mathbb{E}[x_n^2] = \mu^2 + \sigma^2$ であるから：
$$ \frac{1}{N} \sum_{n=1}^N \mathbb{E}[x_n^2] = \frac{1}{N} \sum_{n=1}^N (\mu^2 + \sigma^2) = \mu^2 + \sigma^2 $$

### 2. 第2項の評価
次に、標本平均の二乗の期待値 $\mathbb{E}[\mu_{ML}^2]$ を計算する。
$$ \mathbb{E}[\mu_{ML}^2] = \mathbb{E} \left[ \left( \frac{1}{N} \sum_{n=1}^N x_n \right) \left( \frac{1}{N} \sum_{m=1}^N x_m \right) \right] $$
$$ = \frac{1}{N^2} \sum_{n=1}^N \sum_{m=1}^N \mathbb{E}[x_n x_m] $$
ここで、演習問題 1.12 の結果 $\mathbb{E}[x_n x_m] = \mu^2 + I_{nm} \sigma^2$ を代入する：
$$ = \frac{1}{N^2} \sum_{n=1}^N \sum_{m=1}^N (\mu^2 + I_{nm} \sigma^2) $$
和を分けると、
$$ = \frac{1}{N^2} \left( \sum_{n=1}^N \sum_{m=1}^N \mu^2 \right) + \frac{1}{N^2} \left( \sum_{n=1}^N \sum_{m=1}^N I_{nm} \sigma^2 \right) $$
二重和の第1項は $N^2$ 個の $\mu^2$ の和である。第2項は $I_{nm} = 1$ となる $n=m$ の項のみが残るため、$N$ 個の $\sigma^2$ の和となる。
$$ = \frac{1}{N^2} (N^2 \mu^2) + \frac{1}{N^2} (N \sigma^2) = \mu^2 + \frac{1}{N} \sigma^2 $$

### 3. まとめる
評価した第1項と第2項を元の式に代入する：
$$ \mathbb{E}[\sigma_{ML}^2] = (\mu^2 + \sigma^2) - \left( \mu^2 + \frac{1}{N} \sigma^2 \right) $$
$$ = \sigma^2 - \frac{1}{N} \sigma^2 = \frac{N-1}{N} \sigma^2 $$

以上より、分散の最尤推定量 $\sigma_{ML}^2$ の期待値が $\frac{N-1}{N} \sigma^2$ となることが示された。この結果は、最尤推定量が真の分散を過小評価する（バイアスを持つ）ことを数学的に説明している。


<!-- ORIGINAL GROUPED TEXT:

# 演習問題 1.9

## 問題設定
1次元ガウス分布の分散を求める。具体的には、二次モーメント $\mathbb{E}[x^2]$ を計算し、そこから分散 $var[x]$ が $\sigma^2$ となることを示す。

## 解答
確率変数 $x$ の二次モーメント $\mathbb{E}[x^2]$ は次のように定義される：
$$ \mathbb{E}[x^2] = \int_{-\infty}^{\infty} x^2 \mathcal{N}(x | \mu, \sigma^2) dx $$
$$ \mathbb{E}[x^2] = \int_{-\infty}^{\infty} x^2 \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left\{ -\frac{1}{2\sigma^2}(x - \mu)^2 \right\} dx $$

演習問題1.8と同様に、$y = x - \mu$ （すなわち $x = y + \mu$, $dx = dy$）と変数変換を行う：
$$ \mathbb{E}[x^2] = \int_{-\infty}^{\infty} (y + \mu)^2 \frac{1}{(2\pi\sigma^2)^{1/2}} \exp\left( -\frac{1}{2\sigma^2}y^2 \right) dy $$
展開すると、
$$ (y + \mu)^2 = y^2 + 2\mu y + \mu^2 $$
となるので、積分は以下の3つの項に分割される：
$$ \mathbb{E}[x^2] = \int y^2 \mathcal{N}(y | 0, \sigma^2) dy + 2\mu \int y \mathcal{N}(y | 0, \sigma^2) dy + \mu^2 \int \mathcal{N}(y | 0, \sigma^2) dy $$

各項を評価していく：
1. **第2項**：被積分関数は奇関数であるため、積分値は 0 となる（演習1.8で確認済み）。
2. **第3項**：確率密度関数の全空間での積分は 1 であるため、この項は $\mu^2 \times 1 = \mu^2$ となる。
3. **第1項**：これは平均 0、分散 $\sigma^2$ のガウス分布の分散の定義そのものであるが、部分積分を用いて直接評価する：
   $$ I = \frac{1}{(2\pi\sigma^2)^{1/2}} \int_{-\infty}^{\infty} y \cdot \left( y \exp\left( -\frac{y^2}{2\sigma^2} \right) \right) dy $$
   ここで $u = y$、 $dv = y \exp(-y^2 / 2\sigma^2) dy$ とおくと、 $du = dy$、$v = -\sigma^2 \exp(-y^2 / 2\sigma^2)$ となる。
   部分積分 $\int u dv = [uv] - \int v du$ を適用すると：
   $$ I = \frac{1}{(2\pi\sigma^2)^{1/2}} \left[ -y\sigma^2 \exp\left( -\frac{y^2}{2\sigma^2} \right) \right]_{-\infty}^{\infty} + \frac{1}{(2\pi\sigma^2)^{1/2}} \int_{-\infty}^{\infty} \sigma^2 \exp\left( -\frac{y^2}{2\sigma^2} \right) dy $$
   第1項目の境界値評価では、指数関数の減衰が多項式の増大を上回るため、極限は 0 になる。第2項目は：
   $$ \sigma^2 \int_{-\infty}^{\infty} \mathcal{N}(y | 0, \sigma^2) dy = \sigma^2 \times 1 = \sigma^2 $$

したがって、3つの項をすべて足し合わせると、
$$ \mathbb{E}[x^2] = \sigma^2 + 0 + \mu^2 = \mu^2 + \sigma^2 $$
が得られる。

最後に、分散の定義 $var[x] = \mathbb{E}[x^2] - (\mathbb{E}[x])^2$ を用いる。演習1.8で $\mathbb{E}[x] = \mu$ を示しているので：
$$ var[x] = (\mu^2 + \sigma^2) - \mu^2 = \sigma^2 $$

これにより、パラメータ $\sigma^2$ がこの分布の分散を表すことが厳密に証明された。


In [8]:
# Exercise 1.13 数値検証: 最尤分散のバイアスシミュレーション
np.random.seed(42)
true_sigma2 = 4.0
N_sample = 5
num_trials = 20000

s2_ml_trials = []
for _ in range(num_trials):
    samples = np.random.normal(0, np.sqrt(true_sigma2), N_sample)
    mu_ml = np.mean(samples)
    s2_ml = np.mean((samples - mu_ml)**2)
    s2_ml_trials.append(s2_ml)

mean_s2_ml = np.mean(s2_ml_trials)
expected_theory = ((N_sample - 1) / N_sample) * true_sigma2

print(f"Exercise 1.13 結果:")
print(f"  真の分散: {true_sigma2:.2f}")
print(f"  シミュレーション E[sigma_ML^2]: {mean_s2_ml:.4f}")
print(f"  理論値 (N-1)/N * sigma^2:      {expected_theory:.4f}")
assert np.isclose(mean_s2_ml, expected_theory, rtol=0.02)
print("  => 最尤分散の期待値が (N-1)/N 倍に過小評価される定理を実証！ (OK)")

Exercise 1.13 結果:
  真の分散: 4.00
  シミュレーション E[sigma_ML^2]: 3.1942
  理論値 (N-1)/N * sigma^2:      3.2000
  => 最尤分散の期待値が (N-1)/N 倍に過小評価される定理を実証！ (OK)


## Exercise 1.14

# 演習問題 1.14

## 問題設定
演習問題 1.13 で示された分散の最尤推定量 $\sigma_{ML}^2$ が持つバイアスを補正し、真の分散 $\sigma^2$ の不偏推定量（期待値が真のパラメータに一致する推定量）となるような新しい推定量 $\tilde{\sigma}^2$ を導出する。

## 解答
不偏推定量とは、その期待値が推定したい真のパラメータの値と正確に一致する推定量のことである。すなわち、求める不偏分散 $\tilde{\sigma}^2$ は以下の性質を満たす必要がある：
$$ \mathbb{E}[\tilde{\sigma}^2] = \sigma^2 $$

演習問題 1.13 の結果から、最尤推定量 $\sigma_{ML}^2$ の期待値は次のように表されることが分かっている：
$$ \mathbb{E}[\sigma_{ML}^2] = \frac{N-1}{N} \sigma^2 $$

この式から真の分散 $\sigma^2$ を得るためには、式の両辺に $\frac{N}{N-1}$ を掛ければよい。期待値の線形性（定数倍は外に出せる性質）を利用して変形する：
$$ \frac{N}{N-1} \mathbb{E}[\sigma_{ML}^2] = \sigma^2 $$
$$ \mathbb{E}\left[ \frac{N}{N-1} \sigma_{ML}^2 \right] = \sigma^2 $$

したがって、括弧の中身を新しい推定量 $\tilde{\sigma}^2$ として定義すれば、それは不偏推定量となる。
$$ \tilde{\sigma}^2 = \frac{N}{N-1} \sigma_{ML}^2 $$

ここで、 $\sigma_{ML}^2 = \frac{1}{N} \sum_{n=1}^N (x_n - \mu_{ML})^2$ であるから、これを代入して式を展開する：
$$ \tilde{\sigma}^2 = \frac{N}{N-1} \left( \frac{1}{N} \sum_{n=1}^N (x_n - \mu_{ML})^2 \right) $$
$N$ が相殺されるため、
$$ \tilde{\sigma}^2 = \frac{1}{N-1} \sum_{n=1}^N (x_n - \mu_{ML})^2 $$

### 考察
この結果は、データから分散を推定する際、偏差の平方和をデータ数 $N$ ではなく $N-1$ で割るべきであることを示している。この $N-1$ という値は「自由度」と呼ばれる。
最尤推定量が真の分散を過小評価する理由は、真の平均 $\mu$ ではなく、データ自体から計算された標本平均 $\mu_{ML}$ を用いて偏差を計算しているためである。データはそれ自身の平均値の周りに集まる傾向があるため、偏差の平方和は真の平均の周りの偏差の平方和よりも小さくなってしまう。不偏推定量は、この過小評価を $\frac{N}{N-1}$ という係数で補正したものである。


## Exercise 1.15

# 演習問題 1.15

## 問題設定
$D$次元の入力変数からなる$M$次の多項式において、独立なパラメータ（係数）の数が $\frac{(D+M)!}{M!D!}$ となることを示せ。

## 解答
多項式における各項は、$x_1^{k_1} x_2^{k_2} \cdots x_D^{k_D}$ という形をしている。
ここで、$k_i$ は非負の整数であり、これらの次数の和は最大で $M$ となる。
すなわち、
$$ \sum_{i=1}^D k_i = m \le M $$
を満たす。

この問題は、ダミー変数 $x_0=1$ を導入することで、等式制約に変換できる。
すなわち、$k_0 = M - \sum_{i=1}^D k_i$ とすると、
$$ \sum_{i=0}^D k_i = M $$
となる。
したがって、独立なパラメータの総数は、$M$ 個の対象物を $D+1$ 個の箱に分配する組み合わせの数に等しい。

これは、よく知られた「重複組合せ」の公式を用いて計算できる。
$M$ 個の「丸（オブジェクト）」と $D$ 個の「仕切り」を一列に並べる場合の数を考えればよい。
総数は $M + D$ 個の記号であり、そのうち $M$ 個が丸、$D$ 個が仕切りであるから、
配置の数は以下のようになる：
$$ \binom{M+D}{M} = \frac{(M+D)!}{M!D!} $$
よって、パラメータの数が示された。


## Exercise 1.16 ⭐

# 演習問題 1.16

## 問題設定
対称テンソルにおける独立な要素の数についての帰納法による証明を行う。

## 解答
前問1.15に引き続き、多項式の係数に関する別証を帰納法で与える。
$D$次元、$M$次の多項式の係数の数を $N(D, M)$ とおくと、
$$ N(D, M) = \sum_{m=0}^M \binom{m+D-1}{m} $$
となる。これを用いて、
$$ N(D, M) = \frac{(D+M)!}{D! M!} $$
を証明する。

### 1. $M=0$ または $D=1$ の場合の確認
- $M=0$ の場合：
  定数項のみなので $N(D, 0) = 1$。
  公式では $\frac{D!}{D!0!} = 1$ となり一致する。
- $D=1$ の場合：
  $N(1, M) = \sum_{m=0}^M 1 = M + 1$。
  公式では $\frac{(1+M)!}{1! M!} = M+1$ となり一致する。

### 2. 帰納法のステップ
公式が $N(D, M-1)$ まで成り立つと仮定する。
すなわち、
$$ N(D, M-1) = \frac{(D+M-1)!}{D! (M-1)!} $$
である。
多項式の次数が $M$ になったとき、係数の数は次数 $M$ の項の数だけ増える。
次数 $M$ の項の数は $\binom{M+D-1}{M}$ である。
したがって、
$$ N(D, M) = N(D, M-1) + \frac{(M+D-1)!}{M! (D-1)!} $$
これに帰納法の仮定を代入すると、
$$ N(D, M) = \frac{(D+M-1)!}{D! (M-1)!} + \frac{(M+D-1)!}{M! (D-1)!} $$
共通の項でくくると、
$$ N(D, M) = \frac{(D+M-1)!}{(D-1)! (M-1)!} \left( \frac{1}{D} + \frac{1}{M} \right) $$
$$ = \frac{(D+M-1)!}{(D-1)! (M-1)!} \left( \frac{M+D}{MD} \right) $$
$$ = \frac{(D+M)!}{D! M!} $$
となり、任意の $D, M$ に対して公式が成り立つことが証明された。


## Exercise 1.17

# 演習問題 1.17

## 問題設定
ガンマ関数 $\Gamma(x)$ について、以下の性質を示せ。
1. $\Gamma(x+1) = x\Gamma(x)$
2. $\Gamma(1) = 1$

## 解答

### 1. $\Gamma(x+1) = x\Gamma(x)$ の証明
ガンマ関数は次のように定義される。
$$ \Gamma(x) = \int_0^\infty u^{x-1} e^{-u} du $$

$\Gamma(x+1)$ を定義に従って書き直すと、
$$ \Gamma(x+1) = \int_0^\infty u^x e^{-u} du $$
となる。
これを部分積分を用いて計算する。
$f(u) = u^x$, $g'(u) = e^{-u}$ とおくと、
$f'(u) = x u^{x-1}$, $g(u) = -e^{-u}$ であるから、
$$ \Gamma(x+1) = \left[ -u^x e^{-u} \right]_0^\infty - \int_0^\infty \left( x u^{x-1} \right) (-e^{-u}) du $$

第1項は $u \to \infty$ で $e^{-u}$ が支配的となるため $0$ に収束し、$u=0$ でも $0$ となる（$x>0$のとき）。
したがって、
$$ \Gamma(x+1) = x \int_0^\infty u^{x-1} e^{-u} du = x \Gamma(x) $$
が示された。

### 2. $\Gamma(1) = 1$ の証明
定義に $x=1$ を代入する。
$$ \Gamma(1) = \int_0^\infty u^{1-1} e^{-u} du = \int_0^\infty e^{-u} du $$
これを積分すると、
$$ \Gamma(1) = \left[ -e^{-u} \right]_0^\infty = (0) - (-1) = 1 $$
よって、$\Gamma(1) = 1$ が示された。


## Exercise 1.18

# 演習問題 1.18

## 問題設定
$D$次元空間において、半径 $r$ の球の体積 $V_D(r)$ と表面積 $S_D(r)$ の関係を明らかにし、単位球の体積 $V_D(1)$ と表面積 $S_D(1)$ を求めよ。

## 解答
$D$次元球の体積 $V_D(r)$ は、半径 $r$ の $D$ 乗に比例することが知られている。
単位球の体積を $V_D$ とおくと、
$$ V_D(r) = V_D r^D $$
と書ける。

球の表面積 $S_D(r)$ は、体積 $V_D(r)$ を半径 $r$ で微分したものに等しい。
$$ S_D(r) = \frac{d}{dr} V_D(r) = D V_D r^{D-1} $$
単位球（$r=1$）の表面積を $S_D$ とすると、
$$ S_D = D V_D $$
となる。

次に、$V_D$ を具体的に求めるために、以下の $D$重積分を考える。
$$ I = \int_{-\infty}^\infty \cdots \int_{-\infty}^\infty \exp\left(-\sum_{i=1}^D x_i^2\right) dx_1 \cdots dx_D $$
この積分は各次元について独立に計算でき、ガウス積分の公式により、
$$ I = \left( \int_{-\infty}^\infty \exp(-x^2) dx \right)^D = (\sqrt{\pi})^D = \pi^{D/2} $$

一方で、この積分を極座標系で評価する。
微小体積要素は $S_D(r) dr$ であるから、
$$ I = \int_0^\infty \exp(-r^2) S_D(r) dr = S_D \int_0^\infty r^{D-1} \exp(-r^2) dr $$
ここで $u = r^2$ と変数変換を行うと、$du = 2r dr$ より $dr = \frac{du}{2r}$ となり、
$$ I = S_D \int_0^\infty u^{(D-1)/2} \exp(-u) \frac{du}{2 u^{1/2}} = \frac{S_D}{2} \int_0^\infty u^{D/2 - 1} \exp(-u) du $$
これはガンマ関数の定義そのものであり、
$$ I = \frac{S_D}{2} \Gamma\left(\frac{D}{2}\right) $$
となる。

2つの $I$ の表現を等置して $S_D$ について解くと、
$$ S_D = \frac{2 \pi^{D/2}}{\Gamma(D/2)} $$
さらに、$V_D = S_D / D$ の関係と $\Gamma(D/2 + 1) = (D/2)\Gamma(D/2)$ を用いると、
$$ V_D = \frac{\pi^{D/2}}{(D/2) \Gamma(D/2)} = \frac{\pi^{D/2}}{\Gamma(D/2 + 1)} $$
以上により、$D$次元単位球の表面積と体積が求められた。


## Exercise 1.19 ⭐

# 演習問題 1.19

## 問題設定
$D$ 次元空間における半径 $r$ の球について、大部分の体積が表面付近の薄い殻（シェル）に集中していることを示せ。

## 解答
半径 $r$ の $D$ 次元球の体積を $V(r)$ と表す。前問の結果より、球の体積は半径の $D$ 乗に比例する。
$$ V(r) \propto r^D $$

ここで、球の表面から厚さ $\epsilon$ （$0 < \epsilon \ll r$）の薄い殻を考える。
この殻の体積は、半径 $r$ の球の体積から、半径 $r-\epsilon$ の球の体積を引いたものである。
$$ V_{\text{shell}} = V(r) - V(r-\epsilon) $$

殻の体積が全体積に占める割合（比率）を $f$ とすると、
$$ f = \frac{V(r) - V(r-\epsilon)}{V(r)} = 1 - \frac{V(r-\epsilon)}{V(r)} $$
体積が $r^D$ に比例することを用いると、
$$ f = 1 - \left( \frac{r-\epsilon}{r} \right)^D = 1 - \left( 1 - \frac{\epsilon}{r} \right)^D $$

次元 $D$ が非常に大きい場合を考える。
$$ \lim_{D \to \infty} \left( 1 - \frac{\epsilon}{r} \right)^D $$
$\epsilon > 0$ であるため、$0 < 1 - \frac{\epsilon}{r} < 1$ となる。
したがって、$D$ が大きくなるにつれて、この項は急速に $0$ に近づく。
$$ \lim_{D \to \infty} \left( 1 - \frac{\epsilon}{r} \right)^D = 0 $$

よって、高次元においては
$$ f \approx 1 $$
となり、球の体積の大部分が表面付近の極めて薄い殻に集中することが示された。
これは「次元の呪い (Curse of Dimensionality)」の幾何学的な現れの一つである。


<!-- ORIGINAL GROUPED TEXT:

# 演習問題 1.14

## 問題設定
演習問題 1.13 で示された分散の最尤推定量 $\sigma_{ML}^2$ が持つバイアスを補正し、真の分散 $\sigma^2$ の不偏推定量（期待値が真のパラメータに一致する推定量）となるような新しい推定量 $\tilde{\sigma}^2$ を導出する。

## 解答
不偏推定量とは、その期待値が推定したい真のパラメータの値と正確に一致する推定量のことである。すなわち、求める不偏分散 $\tilde{\sigma}^2$ は以下の性質を満たす必要がある：
$$ \mathbb{E}[\tilde{\sigma}^2] = \sigma^2 $$

演習問題 1.13 の結果から、最尤推定量 $\sigma_{ML}^2$ の期待値は次のように表されることが分かっている：
$$ \mathbb{E}[\sigma_{ML}^2] = \frac{N-1}{N} \sigma^2 $$

この式から真の分散 $\sigma^2$ を得るためには、式の両辺に $\frac{N}{N-1}$ を掛ければよい。期待値の線形性（定数倍は外に出せる性質）を利用して変形する：
$$ \frac{N}{N-1} \mathbb{E}[\sigma_{ML}^2] = \sigma^2 $$
$$ \mathbb{E}\left[ \frac{N}{N-1} \sigma_{ML}^2 \right] = \sigma^2 $$

したがって、括弧の中身を新しい推定量 $\tilde{\sigma}^2$ として定義すれば、それは不偏推定量となる。
$$ \tilde{\sigma}^2 = \frac{N}{N-1} \sigma_{ML}^2 $$

ここで、 $\sigma_{ML}^2 = \frac{1}{N} \sum_{n=1}^N (x_n - \mu_{ML})^2$ であるから、これを代入して式を展開する：
$$ \tilde{\sigma}^2 = \frac{N}{N-1} \left( \frac{1}{N} \sum_{n=1}^N (x_n - \mu_{ML})^2 \right) $$
$N$ が相殺されるため、
$$ \tilde{\sigma}^2 = \frac{1}{N-1} \sum_{n=1}^N (x_n - \mu_{ML})^2 $$

### 考察
この結果は、データから分散を推定する際、偏差の平方和をデータ数 $N$ ではなく $N-1$ で割るべきであることを示している。この $N-1$ という値は「自由度」と呼ばれる。
最尤推定量が真の分散を過小評価する理由は、真の平均 $\mu$ ではなく、データ自体から計算された標本平均 $\mu_{ML}$ を用いて偏差を計算しているためである。データはそれ自身の平均値の周りに集まる傾向があるため、偏差の平方和は真の平均の周りの偏差の平方和よりも小さくなってしまう。不偏推定量は、この過小評価を $\frac{N}{N-1}$ という係数で補正したものである。


In [9]:
# Exercise 1.16 & 1.19 数値シミュレーション
eps = 0.05
dimensions = [1, 2, 5, 10, 20, 50, 100, 200]
shell_ratios = [1.0 - (1.0 - eps)**D for D in dimensions]

print("Exercise 1.16 高次元球の球殻体積比率 (厚み epsilon=0.05):")
for D, r_val in zip(dimensions, shell_ratios):
    print(f"  D = {D:3d}: 球殻体積比率 = {r_val * 100:.2f}%")

# Exercise 1.19 高次元ベクトルのコサイン類似度
cos_sim_means = []
for D in [2, 10, 50, 200, 1000]:
    v1 = np.random.normal(0, 1, (1000, D))
    v2 = np.random.normal(0, 1, (1000, D))
    cos_sim = np.sum(v1 * v2, axis=1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1))
    cos_sim_means.append(np.mean(np.abs(cos_sim)))

print("\nExercise 1.19 高次元ベクトルの平均直交度 |cos theta|:")
for D, c_val in zip([2, 10, 50, 200, 1000], cos_sim_means):
    print(f"  D = {D:4d}: 平均 |cos theta| = {c_val:.4f} (0 に漸近 => 直交)")

Exercise 1.16 高次元球の球殻体積比率 (厚み epsilon=0.05):
  D =   1: 球殻体積比率 = 5.00%
  D =   2: 球殻体積比率 = 9.75%
  D =   5: 球殻体積比率 = 22.62%
  D =  10: 球殻体積比率 = 40.13%
  D =  20: 球殻体積比率 = 64.15%
  D =  50: 球殻体積比率 = 92.31%
  D = 100: 球殻体積比率 = 99.41%
  D = 200: 球殻体積比率 = 100.00%

Exercise 1.19 高次元ベクトルの平均直交度 |cos theta|:
  D =    2: 平均 |cos theta| = 0.6329 (0 に漸近 => 直交)
  D =   10: 平均 |cos theta| = 0.2579 (0 に漸近 => 直交)
  D =   50: 平均 |cos theta| = 0.1163 (0 に漸近 => 直交)
  D =  200: 平均 |cos theta| = 0.0563 (0 に漸近 => 直交)
  D = 1000: 平均 |cos theta| = 0.0247 (0 に漸近 => 直交)


## Exercise 1.20

# 演習問題 1.20

## 問題設定
高次元ガウス分布において、確率質量の大部分が特定の半径を持つ薄い殻に集中していることを示せ。

## 解答
$D$次元の多変量ガウス分布において、平均が $\mathbf{0}$、共分散行列が $\sigma^2 \mathbf{I}$ の場合を考える。
確率密度関数は次のように与えられる。
$$ p(\mathbf{x}) = \frac{1}{(2\pi\sigma^2)^{D/2}} \exp\left(-\frac{\|\mathbf{x}\|^2}{2\sigma^2}\right) $$

原点からの距離を $r = \|\mathbf{x}\|$ とすると、半径 $r$ の点における確率は中心で最大となる。
しかし、確率「質量」（微小体積あたりの確率の積分）を考える必要がある。
極座標に変換すると、微小体積要素は $S_D(r) dr = S_D r^{D-1} dr$ となる（$S_D$は単位球の表面積）。

半径 $r$ にある微小な殻（シェル）の確率質量関数 $p(r)$ は、密度関数と表面積の積となる。
$$ p(r) dr = p(\mathbf{x}) S_D(r) dr = \frac{S_D}{(2\pi\sigma^2)^{D/2}} r^{D-1} \exp\left(-\frac{r^2}{2\sigma^2}\right) dr $$

この関数 $p(r)$ が最大となる半径 $r = \hat{r}$ を求める。
$p(r)$ の対数をとると計算が容易になる。
$$ \ln p(r) = (D-1) \ln r - \frac{r^2}{2\sigma^2} + \text{const} $$
$r$ について微分して $0$ と置く。
$$ \frac{d}{dr} \ln p(r) = \frac{D-1}{r} - \frac{r}{\sigma^2} = 0 $$
$$ \frac{r}{\sigma^2} = \frac{D-1}{r} \implies r^2 = \sigma^2 (D-1) $$
したがって、最大値をとる半径は
$$ \hat{r} = \sigma \sqrt{D-1} $$
となる。

$D$ が大きいとき、$\hat{r} \approx \sigma \sqrt{D}$ となる。
密度関数は原点で最大だが、体積要素 $r^{D-1}$ が急速に増大するため、
両者の積である確率質量関数は $\hat{r} \approx \sigma \sqrt{D}$ 付近で鋭いピークを持つようになる。
これが、高次元ガウス分布の確率質量が「シャボン玉」のように薄い殻に集中する理由である。


## Exercise 1.21

# 演習問題 1.21

## 問題設定
確率変数 $x, y$ において、期待値と分散の性質を用いて、条件付き期待値 $\mathbb{E}[y|x]$ に関する関係式を導け。
特に、予測値 $f(x)$ と真の値 $y$ の二乗誤差の期待値 $\mathbb{E}[(y - f(x))^2]$ を最小化する $f(x)$ が、条件付き期待値 $\mathbb{E}[y|x]$ になることを示せ。

## 解答
二乗誤差の期待値（損失関数）を考える。
$$ L = \mathbb{E}[(y - f(x))^2] = \iint (y - f(x))^2 p(x, y) dx dy $$

乗法定理 $p(x, y) = p(y|x)p(x)$ を用いて書き直すと、
$$ L = \int \left( \int (y - f(x))^2 p(y|x) dy \right) p(x) dx $$

内側の積分（$x$ が与えられたときの条件付き期待値）を最小化することを考える。
$$ \int (y - f(x))^2 p(y|x) dy $$
これを展開して平方完成、または $f(x)$ について微分する。
$f(x)$ で微分して $0$ と置くと、
$$ \frac{\partial}{\partial f(x)} \int (y - f(x))^2 p(y|x) dy = \int -2(y - f(x)) p(y|x) dy = 0 $$
$$ \int y p(y|x) dy - f(x) \int p(y|x) dy = 0 $$

ここで、$\int p(y|x) dy = 1$ であるため、
$$ f(x) = \int y p(y|x) dy = \mathbb{E}[y|x] $$
が得られる。

さらに、二乗誤差の最小値を計算するために $y - f(x)$ を巧みに変形する。
$$ (y - f(x))^2 = (y - \mathbb{E}[y|x] + \mathbb{E}[y|x] - f(x))^2 $$
これを展開すると、
$$ (y - \mathbb{E}[y|x])^2 + 2(y - \mathbb{E}[y|x])(\mathbb{E}[y|x] - f(x)) + (\mathbb{E}[y|x] - f(x))^2 $$
$y$ について条件付き期待値をとると、交差項の $\mathbb{E}[y - \mathbb{E}[y|x]|x]$ は $0$ になる。
したがって、全体の期待値は次のように分解される。
$$ L = \mathbb{E}[(y - \mathbb{E}[y|x])^2] + \mathbb{E}[(\mathbb{E}[y|x] - f(x))^2] $$

第1項は $y$ の条件付き分散（ノイズによる本質的な誤差）であり、$f(x)$ に依存しない。
第2項は $f(x)$ の選び方による誤差であり、非負である。
これを最小にするためには第2項を $0$ にすればよく、すなわち
$$ f(x) = \mathbb{E}[y|x] $$
と選べばよいことが改めて示された。


## Exercise 1.22

# 演習問題 1.22

## 問題文
0-1損失行列（正分類の場合の損失が0、誤分類の場合の損失が1）を用いた場合、期待損失を最小化する決定則が、各 $\mathbf{x}$ に対して事後確率 $p(C_k|\mathbf{x})$ を最大化するクラス $C_k$ を選ぶことと同値になることを示せ。

## 解答と解説
### 1. 期待損失の定式化
一般的な損失行列 $L_{kj}$ における期待損失（リスク）は次のように定義されます。
$$ \mathbb{E}[L] = \sum_{k} \sum_{j} \int_{\mathcal{R}_j} L_{kj} p(\mathbf{x}, C_k) \mathrm{d}\mathbf{x} $$
ここで、$\mathcal{R}_j$ は入力 $\mathbf{x}$ をクラス $C_j$ に割り当てる決定領域を表します。

ベイズの定理より $p(\mathbf{x}, C_k) = p(C_k | \mathbf{x}) p(\mathbf{x})$ であるため、上式は以下のように書き換えられます。
$$ \mathbb{E}[L] = \int \left( \sum_{j} \sum_{k} L_{kj} p(C_k | \mathbf{x}) I(\mathbf{x} \in \mathcal{R}_j) \right) p(\mathbf{x}) \mathrm{d}\mathbf{x} $$
期待損失全体を最小化するためには、被積分関数がすべての $\mathbf{x}$ について最小になればよいことになります。したがって、各 $\mathbf{x}$ について以下の値を最小にするようなクラス $C_j$ を選択すべきです。
$$ \sum_{k} L_{kj} p(C_k | \mathbf{x}) $$

### 2. 0-1 損失行列の適用
問題設定より、0-1損失行列は次のように定義されます。
$$ L_{kj} = 1 - I_{kj} = \begin{cases} 0 & (k = j) \\ 1 & (k \neq j) \end{cases} $$
これを先ほどの最小化すべき式に代入します。
$$ \sum_{k} L_{kj} p(C_k | \mathbf{x}) = \sum_{k} (1 - I_{kj}) p(C_k | \mathbf{x}) $$
$$ = \sum_{k} p(C_k | \mathbf{x}) - \sum_{k} I_{kj} p(C_k | \mathbf{x}) $$
確率の和の性質から $\sum_{k} p(C_k | \mathbf{x}) = 1$ であり、クロネッカーのデルタの性質から $\sum_{k} I_{kj} p(C_k | \mathbf{x}) = p(C_j | \mathbf{x})$ となります。
したがって、最小化すべき式は以下のように簡略化されます。
$$ \sum_{k} L_{kj} p(C_k | \mathbf{x}) = 1 - p(C_j | \mathbf{x}) $$

### 3. 結論
上記の結果より、期待損失を最小化するためには各 $\mathbf{x}$ において $1 - p(C_j | \mathbf{x})$ を最小化するクラス $C_j$ を選べばよいことがわかります。
これはすなわち、**事後確率 $p(C_j | \mathbf{x})$ を最大化するクラス $C_j$ を選ぶこと**と完全に同値です。
したがって、0-1損失関数の下での最適な決定則は、事後確率を最大化するクラスに割り当てることであることが示されました。


## Exercise 1.23

# 演習問題 1.23

## 問題文
期待損失の式
$$ \mathbb{E}[L] = \sum_k \sum_j \int_{\mathcal{R}_j} L_{kj} p(\mathbf{x}, C_k) \mathrm{d}\mathbf{x} $$
において、結合確率 $p(\mathbf{x}, C_k)$ ではなく、クラス条件付き確率 $p(\mathbf{x} | C_k)$ に基づいて決定則を構築する場合を考える。このとき、損失行列 $L_{kj}$ をクラスの事前確率 $p(C_k)$ を用いてどのように修正すれば、期待損失を最小化する最適決定則が等価になるかを示せ。

## 解答と解説
### 1. 期待損失の分解
期待損失は、同時確率密度 $p(\mathbf{x}, C_k)$ を用いて次のように表されます。
$$ \mathbb{E}[L] = \sum_k \sum_j \int_{\mathcal{R}_j} L_{kj} p(\mathbf{x}, C_k) \mathrm{d}\mathbf{x} $$
確率の乗法定理 $p(\mathbf{x}, C_k) = p(\mathbf{x} | C_k) p(C_k)$ を用いると、期待損失は以下のように書き換えられます。
$$ \mathbb{E}[L] = \sum_k \sum_j \int_{\mathcal{R}_j} L_{kj} p(C_k) p(\mathbf{x} | C_k) \mathrm{d}\mathbf{x} $$

### 2. 損失行列の再定義
ここで、修正された新しい損失行列 $\tilde{L}_{kj}$ を次のように定義します。
$$ \tilde{L}_{kj} = L_{kj} p(C_k) $$
これを期待損失の式に代入すると、式は次のように簡潔になります。
$$ \mathbb{E}[L] = \sum_k \sum_j \int_{\mathcal{R}_j} \tilde{L}_{kj} p(\mathbf{x} | C_k) \mathrm{d}\mathbf{x} $$
積分の順序と和の順序を入れ替えることで、次のように整理できます。
$$ \mathbb{E}[L] = \sum_j \int_{\mathcal{R}_j} \left( \sum_k \tilde{L}_{kj} p(\mathbf{x} | C_k) \right) \mathrm{d}\mathbf{x} $$

### 3. 最適な決定則
期待損失全体を最小化するためには、被積分関数である $\sum_k \tilde{L}_{kj} p(\mathbf{x} | C_k)$ を各入力 $\mathbf{x}$ について最小にするようなクラス $C_j$（すなわち領域 $\mathcal{R}_j$）を選択すればよいことになります。

したがって、クラス条件付き確率（尤度）$p(\mathbf{x} | C_k)$ に基づいて決定を行う場合、元の損失行列 $L_{kj}$ に事前確率 $p(C_k)$ を乗じた **$\tilde{L}_{kj} = L_{kj} p(C_k)$ を新たな損失行列として用いることで**、全く等価な期待損失最小化の基準を得ることができます。


## Exercise 1.24 ⭐

# 演習問題 1.24

## 問題文
「リジェクトオプション（棄却オプション）」を含む分類問題を考える。正分類の損失を 0、誤分類の損失を 1、リジェクト（判定保留）の損失を $\lambda$ とする。期待損失を最小化する最適な決定則は、事後確率の最大値が特定の閾値 $\theta$ に満たない場合にリジェクトすることであることを示し、$\lambda$ と $\theta$ の関係を導け。

## 解答と解説
### 1. 期待損失の定式化
ある入力 $\mathbf{x}$ に対してクラス $C_j$ を予測（ラベル付け）した場合の期待損失（条件付きリスク）は次のように計算されます。
$$ \mathbb{E}[L | \mathbf{x}, \text{assign to } C_j] = \sum_k L_{kj} p(C_k | \mathbf{x}) $$
正分類で $L_{jj} = 0$、誤分類で $L_{kj} = 1 \ (k \neq j)$ であるため、
$$ \mathbb{E}[L | \mathbf{x}, \text{assign to } C_j] = \sum_{k \neq j} p(C_k | \mathbf{x}) = 1 - p(C_j | \mathbf{x}) $$
クラスを予測する場合、この損失を最小にするためには、事後確率 $p(C_j | \mathbf{x})$ が最大となるクラス $C_j$ を選択すべきであり、その時の期待損失は以下となります。
$$ \text{Loss}_{\text{assign}} = 1 - \max_j p(C_j | \mathbf{x}) $$

一方で、判定をリジェクトした場合の期待損失は常に $\lambda$ です。
$$ \text{Loss}_{\text{reject}} = \lambda $$

### 2. リジェクト判定の条件
最適な決定則は、予測を行った場合の最小損失 $\text{Loss}_{\text{assign}}$ と、リジェクトした場合の損失 $\text{Loss}_{\text{reject}}$ のうち、より小さい方を選択することです。
したがって、リジェクトが選ばれる条件は次のように表されます。
$$ \text{Loss}_{\text{reject}} < \text{Loss}_{\text{assign}} $$
$$ \lambda < 1 - \max_j p(C_j | \mathbf{x}) $$
これを変形すると以下のようになります。
$$ \max_j p(C_j | \mathbf{x}) < 1 - \lambda $$

### 3. 閾値 $\theta$ との対応関係
問題設定より、事後確率の最大値が閾値 $\theta$ より小さいときにリジェクトを行うとされています。すなわち、リジェクト条件は
$$ \max_j p(C_j | \mathbf{x}) < \theta $$
です。
先ほど導出した式と比較することで、閾値 $\theta$ と損失 $\lambda$ の間には以下の関係が成り立つことがわかります。
$$ \theta = 1 - \lambda $$

**【考察】**
- $\lambda = 0$ の場合（リジェクトにペナルティがない場合）、$\theta = 1$ となり、完全な確信がない限りすべてリジェクトされます。
- $\lambda \ge 1$ の場合（リジェクトの損失が誤分類以上の場合）、$\theta \le 0$ となり、リジェクトオプションは決して選ばれなくなります。


## Exercise 1.25

# 演習問題 1.25

## 問題文
二乗損失関数の期待値
$$ \mathbb{E}[L] = \iint \{y(\mathbf{x}) - t\}^2 p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t $$
を最小化する最適な予測関数 $y(\mathbf{x})$ が、条件付き期待値 $y(\mathbf{x}) = \mathbb{E}[t|\mathbf{x}]$ で与えられることを、変分法（または $y(\mathbf{x})$ による微分）を用いて示せ。

## 解答と解説
### 1. 汎関数積分の微分（変分法）
期待損失関数 $\mathbb{E}[L]$ は $y(\mathbf{x})$ を入力とする汎関数となっています。特定の入力 $\mathbf{x}$ に対する $y(\mathbf{x})$ の最適値を求めるために、被積分関数の中の $\mathbf{x}$ を固定し、各 $\mathbf{x}$ 独立に最小化を行います。

期待損失の式において、各 $\mathbf{x}$ について以下の積分（条件付き期待損失に等比例する部分）を最小化すればよいことが分かります。
$$ f(y(\mathbf{x})) = \int \{y(\mathbf{x}) - t\}^2 p(\mathbf{x}, t) \mathrm{d}t $$

### 2. $y(\mathbf{x})$ による微分と最適解の導出
$f(y(\mathbf{x}))$ を $y(\mathbf{x})$ について微分し、ゼロとおくことで極値を求めます。
$$ \frac{\partial f(y(\mathbf{x}))}{\partial y(\mathbf{x})} = \int 2 \{y(\mathbf{x}) - t\} p(\mathbf{x}, t) \mathrm{d}t = 0 $$
積分を分配すると、
$$ 2 y(\mathbf{x}) \int p(\mathbf{x}, t) \mathrm{d}t - 2 \int t p(\mathbf{x}, t) \mathrm{d}t = 0 $$
ここで周辺化の性質より $\int p(\mathbf{x}, t) \mathrm{d}t = p(\mathbf{x})$ です。したがって、
$$ y(\mathbf{x}) p(\mathbf{x}) = \int t p(\mathbf{x}, t) \mathrm{d}t $$
両辺を $p(\mathbf{x})$ で割ると（$p(\mathbf{x}) > 0$ を仮定）、
$$ y(\mathbf{x}) = \frac{\int t p(\mathbf{x}, t) \mathrm{d}t}{p(\mathbf{x})} = \int t \frac{p(\mathbf{x}, t)}{p(\mathbf{x})} \mathrm{d}t $$
乗法定理 $p(t | \mathbf{x}) = p(\mathbf{x}, t) / p(\mathbf{x})$ より、
$$ y(\mathbf{x}) = \int t p(t | \mathbf{x}) \mathrm{d}t = \mathbb{E}[t | \mathbf{x}] $$

### 3. 結論
これにより、二乗損失関数の下での最適な関数 $y(\mathbf{x})$ は、入力 $\mathbf{x}$ が与えられたときの目標値 $t$ の条件付き期待値（回帰関数）となることが厳密に示されました。


## Exercise 1.26

# 演習問題 1.26

## 問題文
二乗損失関数の期待損失 $\mathbb{E}[L]$ に対して展開を行い、次の分解（テキストの式1.90）を導出せよ。
$$ \mathbb{E}[L] = \int \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}^2 p(\mathbf{x}) \mathrm{d}\mathbf{x} + \int \{\mathbb{E}[t|\mathbf{x}] - t\}^2 p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t $$

## 解答と解説
### 1. 二乗誤差項の展開
期待損失関数の定義は以下の通りです。
$$ \mathbb{E}[L] = \iint \{y(\mathbf{x}) - t\}^2 p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t $$
被積分関数の中の二乗項に、意図的に条件付き期待値 $\mathbb{E}[t|\mathbf{x}]$ を足して引きます。
$$ \{y(\mathbf{x}) - t\}^2 = \{ (y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]) + (\mathbb{E}[t|\mathbf{x}] - t) \}^2 $$
これを展開すると、以下の3つの項が得られます。
$$ \{y(\mathbf{x}) - t\}^2 = \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}^2 + \{\mathbb{E}[t|\mathbf{x}] - t\}^2 + 2\{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}\{\mathbb{E}[t|\mathbf{x}] - t\} $$

### 2. 各項の積分
この展開を元の期待損失の式に代入し、各項について積分を計算します。

**第1項:**
$$ \iint \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}^2 p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t = \int \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}^2 \left( \int p(\mathbf{x}, t) \mathrm{d}t \right) \mathrm{d}\mathbf{x} $$
$\int p(\mathbf{x}, t) \mathrm{d}t = p(\mathbf{x})$ より、
$$ = \int \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}^2 p(\mathbf{x}) \mathrm{d}\mathbf{x} $$

**第2項:**
これはそのまま残ります。
$$ \iint \{\mathbb{E}[t|\mathbf{x}] - t\}^2 p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t $$

**第3項（交差項）:**
$$ 2 \iint \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}\{\mathbb{E}[t|\mathbf{x}] - t\} p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t $$
ここで、$p(\mathbf{x}, t) = p(t | \mathbf{x}) p(\mathbf{x})$ を用いて内側の $t$ についての積分に注目します。$y(\mathbf{x})$ および $\mathbb{E}[t|\mathbf{x}]$ は $t$ に依存しないため積分の外に出せます。
$$ = 2 \int \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\} p(\mathbf{x}) \left[ \int \{\mathbb{E}[t|\mathbf{x}] - t\} p(t | \mathbf{x}) \mathrm{d}t \right] \mathrm{d}\mathbf{x} $$
大カッコの中の積分を評価します。
$$ \int \mathbb{E}[t|\mathbf{x}] p(t | \mathbf{x}) \mathrm{d}t - \int t p(t | \mathbf{x}) \mathrm{d}t = \mathbb{E}[t|\mathbf{x}] \cdot 1 - \mathbb{E}[t|\mathbf{x}] = 0 $$
したがって、交差項全体が 0 となることが分かります。

### 3. 結論
交差項が 0 になるため、残った第1項と第2項を足し合わせることで、目的の式が導出されます。
$$ \mathbb{E}[L] = \int \{y(\mathbf{x}) - \mathbb{E}[t|\mathbf{x}]\}^2 p(\mathbf{x}) \mathrm{d}\mathbf{x} + \iint \{\mathbb{E}[t|\mathbf{x}] - t\}^2 p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t $$
この式の第1項は予測モデル $y(\mathbf{x})$ の依存部分であり、$y(\mathbf{x}) = \mathbb{E}[t|\mathbf{x}]$ で最小値 0 を取ります。第2項はデータそのものが持つ本質的なノイズの分散を表しており、$y(\mathbf{x})$ に依存しない達成不可能な最小誤差（ベイズ誤差）を示しています。


## Exercise 1.27

# 演習問題 1.27

## 問題文
ミンコフスキー損失関数に基づく期待損失
$$ \mathbb{E}[L_q] = \iint |y(\mathbf{x}) - t|^q p(\mathbf{x}, t) \mathrm{d}\mathbf{x} \mathrm{d}t $$
について考える。
(a) $q=1$ の場合、これを最小化する $y(\mathbf{x})$ が条件付き中央値（conditional median）になることを示せ。
(b) $q \to 0$ の場合、これを最小化する $y(\mathbf{x})$ が条件付き最頻値（conditional mode）になることを示せ。

## 解答と解説
### (a) $q=1$ の場合（絶対値損失と中央値）
$q=1$ のとき、期待損失の $y(\mathbf{x})$ に関する最小化は、各 $\mathbf{x}$ ごとに次の関数 $f(y)$ を最小化することに等しいです（ここで $\mathbf{x}$ への依存関係を省略して $y$ と書きます）。
$$ f(y) = \int_{-\infty}^{\infty} |y - t| p(t | \mathbf{x}) \mathrm{d}t $$
絶対値を外すため、積分区間を $t < y$ と $t \ge y$ に分割します。
$$ f(y) = \int_{-\infty}^{y} (y - t) p(t | \mathbf{x}) \mathrm{d}t - \int_{y}^{\infty} (y - t) p(t | \mathbf{x}) \mathrm{d}t $$
$y$ で微分するためにライプニッツの積分法則を適用します。
$$ \frac{\partial f(y)}{\partial y} = \int_{-\infty}^{y} p(t | \mathbf{x}) \mathrm{d}t + (y - y) p(y | \mathbf{x}) - \int_{y}^{\infty} p(t | \mathbf{x}) \mathrm{d}t - (- (y - y) p(y | \mathbf{x})) $$
$$ = \int_{-\infty}^{y} p(t | \mathbf{x}) \mathrm{d}t - \int_{y}^{\infty} p(t | \mathbf{x}) \mathrm{d}t = 0 $$
これを満たすためには、以下が成り立つ必要があります。
$$ \int_{-\infty}^{y} p(t | \mathbf{x}) \mathrm{d}t = \int_{y}^{\infty} p(t | \mathbf{x}) \mathrm{d}t $$
確率密度の全体積分は 1 であるため、この等式は双方が $1/2$ になることを意味します。すなわち、確率の累積がちょうど 50% になる点であり、これは**条件付き中央値**の定義そのものです。

### (b) $q \to 0$ の場合（0-1近似と最頻値）
期待損失を次のように変形します。
$$ \mathbb{E}[L_q] = \int p(\mathbf{x}) \left( \int |y(\mathbf{x}) - t|^q p(t | \mathbf{x}) \mathrm{d}t \right) \mathrm{d}\mathbf{x} $$
内側の積分 $\int |y(\mathbf{x}) - t|^q p(t | \mathbf{x}) \mathrm{d}t$ を最小化することを考えます。
$q \to 0$ の極限では、関数 $|y - t|^q$ は次のように振る舞います。
$$ \lim_{q \to 0} |y - t|^q = \begin{cases} 0 & (t = y) \\ 1 & (t \neq y) \end{cases} $$
厳密な極限では積分値の差が見えづらいため、$t = y$ の非常に小さな $\epsilon$ 近傍 $\mathcal{U} = [y-\epsilon, y+\epsilon]$ を考えます。$q$ が非常に0に近いとき、
$$ \int |y - t|^q p(t | \mathbf{x}) \mathrm{d}t \approx \int_{t \notin \mathcal{U}} 1 \cdot p(t | \mathbf{x}) \mathrm{d}t + \int_{t \in \mathcal{U}} 0 \cdot p(t | \mathbf{x}) \mathrm{d}t = 1 - \int_{y-\epsilon}^{y+\epsilon} p(t | \mathbf{x}) \mathrm{d}t $$
この値を最小化することは、第二項である積分 $\int_{y-\epsilon}^{y+\epsilon} p(t | \mathbf{x}) \mathrm{d}t$ を最大化することと同義です。
$\epsilon$ が十分に小さいとき、この積分は $2\epsilon \cdot p(y | \mathbf{x})$ で近似されます。したがって、この値を最大化する $y$ は、確率密度関数 $p(t | \mathbf{x})$ が最大となる点、すなわち**条件付き最頻値（conditional mode）**になります。


<!-- ORIGINAL GROUPED TEXT:

# 演習問題 1.20

## 問題設定
高次元ガウス分布において、確率質量の大部分が特定の半径を持つ薄い殻に集中していることを示せ。

## 解答
$D$次元の多変量ガウス分布において、平均が $\mathbf{0}$、共分散行列が $\sigma^2 \mathbf{I}$ の場合を考える。
確率密度関数は次のように与えられる。
$$ p(\mathbf{x}) = \frac{1}{(2\pi\sigma^2)^{D/2}} \exp\left(-\frac{\|\mathbf{x}\|^2}{2\sigma^2}\right) $$

原点からの距離を $r = \|\mathbf{x}\|$ とすると、半径 $r$ の点における確率は中心で最大となる。
しかし、確率「質量」（微小体積あたりの確率の積分）を考える必要がある。
極座標に変換すると、微小体積要素は $S_D(r) dr = S_D r^{D-1} dr$ となる（$S_D$は単位球の表面積）。

半径 $r$ にある微小な殻（シェル）の確率質量関数 $p(r)$ は、密度関数と表面積の積となる。
$$ p(r) dr = p(\mathbf{x}) S_D(r) dr = \frac{S_D}{(2\pi\sigma^2)^{D/2}} r^{D-1} \exp\left(-\frac{r^2}{2\sigma^2}\right) dr $$

この関数 $p(r)$ が最大となる半径 $r = \hat{r}$ を求める。
$p(r)$ の対数をとると計算が容易になる。
$$ \ln p(r) = (D-1) \ln r - \frac{r^2}{2\sigma^2} + \text{const} $$
$r$ について微分して $0$ と置く。
$$ \frac{d}{dr} \ln p(r) = \frac{D-1}{r} - \frac{r}{\sigma^2} = 0 $$
$$ \frac{r}{\sigma^2} = \frac{D-1}{r} \implies r^2 = \sigma^2 (D-1) $$
したがって、最大値をとる半径は
$$ \hat{r} = \sigma \sqrt{D-1} $$
となる。

$D$ が大きいとき、$\hat{r} \approx \sigma \sqrt{D}$ となる。
密度関数は原点で最大だが、体積要素 $r^{D-1}$ が急速に増大するため、
両者の積である確率質量関数は $\hat{r} \approx \sigma \sqrt{D}$ 付近で鋭いピークを持つようになる。
これが、高次元ガウス分布の確率質量が「シャボン玉」のように薄い殻に集中する理由である。


In [10]:
# Exercise 1.26 数値検証: L1 損失と中央値 / L2 損失と平均
# 外れ値を含む非対称な t の分布: 90% N(0, 1) + 10% N(10, 1)
np.random.seed(42)
t_data = np.concatenate([np.random.normal(0, 1, 9000), np.random.normal(10, 1, 1000)])

y_grid = np.linspace(-1, 5, 500)
loss_l2 = [np.mean((y_val - t_data)**2) for y_val in y_grid]
loss_l1 = [np.mean(np.abs(y_val - t_data)) for y_val in y_grid]

opt_y_l2 = y_grid[np.argmin(loss_l2)]
opt_y_l1 = y_grid[np.argmin(loss_l1)]

print(f"Exercise 1.26 結果:")
print(f"  データ平均値 (Mean):     {np.mean(t_data):.4f} <=> 最適 L2 予測値: {opt_y_l2:.4f}")
print(f"  データ中央値 (Median):   {np.median(t_data):.4f} <=> 最適 L1 予測値: {opt_y_l1:.4f}")
assert np.isclose(opt_y_l2, np.mean(t_data), atol=0.02)
assert np.isclose(opt_y_l1, np.median(t_data), atol=0.02)
print("  => L2損失は平均値、L1損失は中央値を最小点とすることを数値証明！ (OK)")

Exercise 1.26 結果:
  データ平均値 (Mean):     0.9979 <=> 最適 L2 予測値: 0.9960
  データ中央値 (Median):   0.1393 <=> 最適 L1 予測値: 0.1423
  => L2損失は平均値、L1損失は中央値を最小点とすることを数値証明！ (OK)


## Exercise 1.28

# 演習問題 1.28

## 問題文
事象の確率 $p$ のみに依存する情報量（自己情報量）を $h(p)$ とする。独立な2つの事象が同時に起こる場合の情報量が、それぞれの情報量の和になること（加法性）を仮定する。すなわち、
$$ h(p_1 p_2) = h(p_1) + h(p_2) $$
が成り立つとする。この関数方程式を満たす連続関数 $h(p)$ が、定数倍を除いて対数関数 $\ln p$ になること（$h(p) \propto \ln p$）を示せ。

## 解答と解説
### 1. 関数方程式の性質（整数および有理数の場合）
与えられた関数方程式は以下の形をしています。
$$ h(xy) = h(x) + h(y) \quad (0 < x, y \le 1) $$
まず、任意の正の整数 $n$ に対して、帰納法により次が成り立つことが示せます。
$$ h(x^n) = n h(x) $$
（証明：$n=1$ は自明。$n=2$ のとき $h(x^2) = h(xx) = h(x)+h(x) = 2h(x)$。$n=k$ で成立すると仮定すると、$h(x^{k+1}) = h(x^k x) = h(x^k) + h(x) = k h(x) + h(x) = (k+1)h(x)$ となり成立。）

次に、任意の正の有理数 $r = \frac{m}{n}$ （$m, n$ は正の整数）についても調べます。
$y = x^{\frac{m}{n}}$ と置きます。両辺を $n$ 乗すると $y^n = x^m$ です。
関数方程式より、
$$ h(y^n) = h(x^m) \implies n h(y) = m h(x) \implies h(y) = \frac{m}{n} h(x) $$
したがって、有理数 $r$ においても $h(x^r) = r h(x)$ が成り立ちます。

### 2. 連続性による実数への拡張
問題の仮定（または物理的・情報的要請）として $h(p)$ は確率 $p$ に関して連続な関数です。
有理数は実数の中に稠密に存在するため、任意の正の実数 $a$ に対して $r_k \to a$ となる有理数列 $\{r_k\}$ をとることができます。
連続関数であれば極限と関数の適用を交換できるため、任意の実数 $a$ に対して、
$$ h(x^a) = h(\lim_{k \to \infty} x^{r_k}) = \lim_{k \to \infty} h(x^{r_k}) = \lim_{k \to \infty} r_k h(x) = a h(x) $$
が成り立ちます。

### 3. 対数関数との関係の導出
関数が $h(x^a) = a h(x)$ を満たすことがわかりました。
ここで、底をネイピア数 $e$ とした指数表現を考えます。任意の確率 $p \ (0 < p \le 1)$ を $p = e^{\ln p}$ と書くことができます。
これを先ほどの実数乗の公式に適用します。$x = e$, $a = \ln p$ と置くと、
$$ h(p) = h(e^{\ln p}) = (\ln p) \cdot h(e) $$
ここで、$h(e)$ は変数 $p$ に依存しない単なる定数です。これを $C = h(e)$ と置くと、
$$ h(p) = C \ln p $$
と表せます。情報量は常に非負であり、確率が小さいほど情報量は大きくなるべきなので、$h(p) \ge 0$ を満たすために定数 $C$ は負の定数となります（通常 $C = -1$ などとします）。

これにより、関数 $h(p)$ は $\ln p$ に比例すること、すなわち $h(p) \propto \ln p$ が証明されました。


## Exercise 1.29

# 演習問題 1.29

## 問題の概要
離散確率変数 $x$ のエントロピー $H[x]$ が、 $x$ のすべての状態が等確率で起こる一様分布のときに最大化されることを、ラグランジュの未定乗数法を用いて証明する。

## 証明
離散確率変数 $x$ が $M$ 個の状態をとるとする。それぞれの状態をとる確率を $p(x_i) = p_i$ とすると、エントロピーは次のように定義される。
$$
H[x] = -\sum_{i=1}^{M} p_i \ln p_i
$$
ここで、確率の総和は $1$ にならなければならないという制約条件がある。
$$
\sum_{i=1}^{M} p_i = 1
$$
この制約条件のもとで $H[x]$ を最大化するために、ラグランジュ乗数 $\lambda$ を導入し、ラグランジュ関数 $L$ を次のように定義する。
$$
L(p_1, \dots, p_M, \lambda) = -\sum_{i=1}^{M} p_i \ln p_i + \lambda \left( \sum_{i=1}^{M} p_i - 1 \right)
$$
この関数を各 $p_i$ で偏微分して $0$ と置く。
$$
\frac{\partial L}{\partial p_i} = - \ln p_i - 1 + \lambda = 0
$$
これを $p_i$ について解くと、
$$
\ln p_i = \lambda - 1 \implies p_i = \exp(\lambda - 1)
$$
となる。これはすべての $i$ について $p_i$ が同じ定数値をとることを意味している。つまり、
$$
p_1 = p_2 = \dots = p_M
$$
である。制約条件 $\sum_{i=1}^{M} p_i = 1$ を適用すると、
$$
\sum_{i=1}^{M} \exp(\lambda - 1) = M \exp(\lambda - 1) = 1
$$
$$
\exp(\lambda - 1) = \frac{1}{M}
$$
したがって、各状態の確率は
$$
p_i = \frac{1}{M}
$$
となる。これは一様分布に他ならない。

一様分布のときのエントロピーの最大値は、
$$
H[x] = -\sum_{i=1}^{M} \frac{1}{M} \ln \frac{1}{M} = - M \left( \frac{1}{M} \ln \frac{1}{M} \right) = \ln M
$$
となる。以上より、離散確率変数のエントロピーは一様分布のときに最大値 $\ln M$ をとることが示された。


## Exercise 1.30

# 演習問題 1.30

## 問題の概要
2つの1次元ガウス分布 $p(x) = \mathcal{N}(x|\mu, \sigma^2)$ と $q(x) = \mathcal{N}(x|m, s^2)$ の間のカルバック・ライブラー (Kullback-Leibler) 情報量 $\text{KL}(p||q)$ を計算する。

## 証明
カルバック・ライブラー情報量は次のように定義される。
$$
\text{KL}(p||q) = \int_{-\infty}^{\infty} p(x) \ln \frac{p(x)}{q(x)} dx = \int_{-\infty}^{\infty} p(x) \ln p(x) dx - \int_{-\infty}^{\infty} p(x) \ln q(x) dx
$$

ここで、$p(x)$ と $q(x)$ はそれぞれ次のガウス分布である。
$$
p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{(x-\mu)^2}{2\sigma^2} \right)
$$
$$
q(x) = \frac{1}{\sqrt{2\pi s^2}} \exp\left( -\frac{(x-m)^2}{2s^2} \right)
$$

第1項は $-H[p]$ （エントロピーの負の値）に等しい。1次元ガウス分布 $p(x)$ のエントロピーは既知の結果を用いて以下のように表される。
$$
\int p(x) \ln p(x) dx = -H[p] = -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{1}{2}
$$

次に、第2項を計算する。
$$
\ln q(x) = -\frac{1}{2}\ln(2\pi s^2) - \frac{(x-m)^2}{2s^2}
$$
これを $p(x)$ で期待値をとる。
$$
\int p(x) \ln q(x) dx = \int p(x) \left( -\frac{1}{2}\ln(2\pi s^2) - \frac{(x-m)^2}{2s^2} \right) dx
$$
$$
= -\frac{1}{2}\ln(2\pi s^2) - \frac{1}{2s^2} \int p(x) (x-m)^2 dx
$$

積分部分を展開して期待値を計算する。$(x-m)^2 = (x-\mu + \mu-m)^2 = (x-\mu)^2 + 2(x-\mu)(\mu-m) + (\mu-m)^2$ と分解できる。
$p(x)$ において、$\mathbb{E}[x-\mu] = 0$ および $\mathbb{E}[(x-\mu)^2] = \sigma^2$ であるから、
$$
\int p(x) (x-m)^2 dx = \sigma^2 + (\mu-m)^2
$$
となる。したがって、
$$
\int p(x) \ln q(x) dx = -\frac{1}{2}\ln(2\pi s^2) - \frac{\sigma^2 + (\mu-m)^2}{2s^2}
$$

以上をまとめて KL 情報量を計算する。
$$
\text{KL}(p||q) = \left( -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{1}{2} \right) - \left( -\frac{1}{2}\ln(2\pi s^2) - \frac{\sigma^2 + (\mu-m)^2}{2s^2} \right)
$$
整理すると、
$$
\text{KL}(p||q) = \frac{1}{2} \ln \frac{s^2}{\sigma^2} + \frac{\sigma^2 + (\mu-m)^2}{2s^2} - \frac{1}{2}
$$
これが求める結果である。


## Exercise 1.31

# 演習問題 1.31

## 問題の概要
結合確率分布 $p(x, y)$ を持つ2つの確率変数 $x$ と $y$ について、結合エントロピー $H[x, y]$ が条件付きエントロピー $H[y|x]$ と周辺エントロピー $H[x]$ の和で表されること（連鎖律）を証明する。
すなわち、$H[x, y] = H[y|x] + H[x]$ を示す。

## 証明
結合エントロピー $H[x, y]$ の定義は以下の通りである。
$$
H[x, y] = -\iint p(x, y) \ln p(x, y) dx dy
$$
（離散変数の場合は積分の代わりに和の記号となるが、本質的な計算は同じである。）

確率の乗法定理により、結合確率は条件付き確率を用いて次のように分解できる。
$$
p(x, y) = p(y|x)p(x)
$$
対数をとると、
$$
\ln p(x, y) = \ln p(y|x) + \ln p(x)
$$
となる。これを結合エントロピーの式に代入する。
$$
H[x, y] = -\iint p(x, y) \left( \ln p(y|x) + \ln p(x) \right) dx dy
$$
積分を2つの項に分ける。
$$
H[x, y] = -\iint p(x, y) \ln p(y|x) dx dy - \iint p(x, y) \ln p(x) dx dy
$$

第1項について：
$$
-\iint p(x, y) \ln p(y|x) dx dy = H[y|x]
$$
これは条件付きエントロピー $H[y|x]$ の定義そのものである。

第2項について：
$y$ について周辺化すると $\int p(x, y) dy = p(x)$ となることを利用する。
$$
-\iint p(x, y) \ln p(x) dx dy = -\int \left( \int p(x, y) dy \right) \ln p(x) dx = -\int p(x) \ln p(x) dx
$$
これは $x$ のエントロピー $H[x]$ の定義である。

したがって、
$$
H[x, y] = H[y|x] + H[x]
$$
が成り立つことが示された。


## Exercise 1.32

# 演習問題 1.32

## 問題の概要
結合確率分布 $p(x, y)$ を持つ2つの確率変数 $x, y$ において、結合エントロピーについて次の不等式が成り立つことを証明する。
$$
H[x, y] \le H[x] + H[y]
$$
また、等号が成立するための必要十分条件が $x$ と $y$ が独立であること（すなわち $p(x, y) = p(x)p(y)$）であることを示す。

## 証明
カルバック・ライブラー情報量（KL divergence）の非負性を利用する。
任意の2つの確率分布 $p(\mathbf{z})$ と $q(\mathbf{z})$ に対して、$\text{KL}(p||q) \ge 0$ が成り立ち、等号は $p(\mathbf{z}) = q(\mathbf{z})$ のときに限って成立することが知られている（ギブスの不等式）。

ここで、結合分布 $p(x, y)$ と、それぞれの周辺分布の積 $p(x)p(y)$ を考え、これらの間のKL情報量（相互情報量 $I(x, y)$）を計算する。
$$
I(x, y) = \text{KL}(p(x,y)||p(x)p(y)) = \iint p(x, y) \ln \frac{p(x, y)}{p(x)p(y)} dx dy
$$
対数の性質を用いて分解すると、
$$
I(x, y) = \iint p(x, y) \ln p(x, y) dx dy - \iint p(x, y) \ln p(x) dx dy - \iint p(x, y) \ln p(y) dx dy
$$
第1項は $-H[x, y]$ である。第2項と第3項はそれぞれ周辺分布の性質を用いて以下のように計算できる。
$$
- \iint p(x, y) \ln p(x) dx dy = - \int \left( \int p(x, y) dy \right) \ln p(x) dx = - \int p(x) \ln p(x) dx = H[x]
$$
$$
- \iint p(x, y) \ln p(y) dx dy = - \int \left( \int p(x, y) dx \right) \ln p(y) dy = - \int p(y) \ln p(y) dy = H[y]
$$
よって、
$$
I(x, y) = -H[x, y] + H[x] + H[y]
$$
となる。KL情報量の非負性 $\text{KL}(p||q) \ge 0$ より $I(x, y) \ge 0$ であるため、
$$
-H[x, y] + H[x] + H[y] \ge 0
$$
$$
H[x, y] \le H[x] + H[y]
$$
が導かれる。

さらに、等号が成立するのはKL情報量が $0$ になるとき、すなわち
$$
p(x, y) = p(x)p(y)
$$
がすべての $x, y$ について成り立つときのみである。これは $x$ と $y$ が互いに独立であることの定義に他ならない。
以上より、題意は示された。


## Exercise 1.33

# 演習問題 1.33

## 問題の概要
2つの確率変数 $x$ と $y$ の相互情報量 $I(x, y)$ が、エントロピーと条件付きエントロピーを用いて次のように表されることを証明する。
$$
I(x, y) = H[x] - H[x|y] = H[y] - H[y|x]
$$

## 証明
相互情報量 $I(x, y)$ の定義は以下の通りである。
$$
I(x, y) = \iint p(x, y) \ln \frac{p(x, y)}{p(x)p(y)} dx dy
$$
対数の性質を用いて、対数部分を展開する。
$$
\ln \frac{p(x, y)}{p(x)p(y)} = \ln p(x, y) - \ln p(x) - \ln p(y)
$$
また、確率の乗法定理 $p(x, y) = p(y|x)p(x) = p(x|y)p(y)$ より、対数部分は次のように変形することもできる。
$$
\ln \frac{p(x, y)}{p(x)p(y)} = \ln \frac{p(x|y)p(y)}{p(x)p(y)} = \ln \frac{p(x|y)}{p(x)} = \ln p(x|y) - \ln p(x)
$$
これを相互情報量の定義式に代入する。
$$
I(x, y) = \iint p(x, y) \left( \ln p(x|y) - \ln p(x) \right) dx dy
$$
積分を分けると、
$$
I(x, y) = \iint p(x, y) \ln p(x|y) dx dy - \iint p(x, y) \ln p(x) dx dy
$$
第1項は、定義より $-H[x|y]$（条件付きエントロピーの負）である。
第2項は、$y$ について周辺化することで $-H[x]$ となる。
$$
-\iint p(x, y) \ln p(x) dx dy = -\int \left( \int p(x, y) dy \right) \ln p(x) dx = -\int p(x) \ln p(x) dx = H[x]
$$
したがって、
$$
I(x, y) = -H[x|y] + H[x] = H[x] - H[x|y]
$$
が成り立つ。

全く同様にして、対数部分を $p(x, y) = p(y|x)p(x)$ を用いて変形すると、
$$
\ln \frac{p(x, y)}{p(x)p(y)} = \ln \frac{p(y|x)p(x)}{p(x)p(y)} = \ln \frac{p(y|x)}{p(y)} = \ln p(y|x) - \ln p(y)
$$
となるため、これを用いて積分すると、
$$
I(x, y) = \iint p(x, y) \left( \ln p(y|x) - \ln p(y) \right) dx dy
$$
$$
= \iint p(x, y) \ln p(y|x) dx dy - \iint p(x, y) \ln p(y) dx dy = -H[y|x] + H[y] = H[y] - H[y|x]
$$
となる。
以上より、相互情報量が周辺エントロピーから条件付きエントロピーを引いたものに等しいことが証明された。


## Exercise 1.34 ⭐

# 演習問題 1.34

## 問題の概要
連続確率変数 $x$ の微分エントロピーを最大化する分布が、平均 $\mu$ と分散 $\sigma^2$ が与えられた制約のもとではガウス分布になることを、ラグランジュの未定乗数法を用いて証明する。

## 証明
連続変数 $x$ の分布 $p(x)$ に対する微分エントロピー $H[x]$ は次のように定義される。
$$
H[x] = -\int_{-\infty}^{\infty} p(x) \ln p(x) dx
$$
このエントロピーを以下の3つの制約条件のもとで最大化する。
1. 確率分布の正規化条件：
$$ \int_{-\infty}^{\infty} p(x) dx = 1 $$
2. 平均が $\mu$ であること：
$$ \int_{-\infty}^{\infty} x p(x) dx = \mu $$
3. 分散が $\sigma^2$ であること：
$$ \int_{-\infty}^{\infty} (x-\mu)^2 p(x) dx = \sigma^2 $$

ラグランジュの未定乗数 $\lambda_1, \lambda_2, \lambda_3$ を導入し、ラグランジュ関数（汎関数）を次のように設定する。
$$
L(p) = -\int p(x) \ln p(x) dx + \lambda_1 \left( \int p(x) dx - 1 \right) + \lambda_2 \left( \int x p(x) dx - \mu \right) + \lambda_3 \left( \int (x-\mu)^2 p(x) dx - \sigma^2 \right)
$$
変分法を用いて、$p(x)$ に対する $L$ の変分（関数微分）を $0$ と置く。
$$
\frac{\delta L}{\delta p(x)} = -\ln p(x) - 1 + \lambda_1 + \lambda_2 x + \lambda_3 (x-\mu)^2 = 0
$$
これを $p(x)$ について解くと、
$$
p(x) = \exp \left( -1 + \lambda_1 + \lambda_2 x + \lambda_3 (x-\mu)^2 \right)
$$
となる。これを平方完成を用いて整理すると、ガウス関数の形になることが予想できる。指数部分を $x-\mu$ で整理するため、$\lambda_2 x = \lambda_2 (x-\mu) + \lambda_2 \mu$ と書き換えると、
$$
p(x) = \exp \left( \lambda_3 (x-\mu)^2 + \lambda_2 (x-\mu) + (-1 + \lambda_1 + \lambda_2 \mu) \right)
$$
ガウス分布の形 $p(x) \propto \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$ と比較すると、対称性（制約 2 と 3）から $x-\mu$ の1次の項は不要であるため、$\lambda_2 = 0$ となることがわかる。（厳密には、$\int x p(x) dx = \mu$ の制約を満たすために $\lambda_2 = 0$ が要求される。）
したがって、
$$
p(x) = \exp \left( \lambda_3 (x-\mu)^2 + \lambda_1 - 1 \right)
$$
制約条件を満たすように定数 $\lambda_3$ と $\lambda_1$ を決定する。正規化条件と分散の条件より、$\lambda_3 = -\frac{1}{2\sigma^2}$ となり、正規化係数は $\exp(\lambda_1 - 1) = \frac{1}{\sqrt{2\pi\sigma^2}}$ となる。
これを代入すると、
$$
p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp \left( -\frac{(x-\mu)^2}{2\sigma^2} \right)
$$
が得られる。これは平均 $\mu$、分散 $\sigma^2$ のガウス分布 $\mathcal{N}(x|\mu, \sigma^2)$ そのものである。
以上より、与えられた平均と分散の制約のもとで微分エントロピーを最大化するのはガウス分布であることが示された。


## Exercise 1.35

# 演習問題 1.35

## 問題の概要
多変量ガウス分布の微分エントロピーを計算し、次の式になることを証明する。
$$
H[\mathbf{x}] = \frac{1}{2}\ln|\boldsymbol{\Sigma}| + \frac{D}{2}(1 + \ln(2\pi))
$$

## 証明
$D$ 次元の多変量ガウス分布 $p(\mathbf{x}) = \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}, \boldsymbol{\Sigma})$ は次のように定義される。
$$
p(\mathbf{x}) = \frac{1}{(2\pi)^{D/2}|\boldsymbol{\Sigma}|^{1/2}} \exp\left( -\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) \right)
$$

微分エントロピー $H[\mathbf{x}]$ は定義より次のように計算される。
$$
H[\mathbf{x}] = -\int p(\mathbf{x}) \ln p(\mathbf{x}) d\mathbf{x} = -\mathbb{E}[\ln p(\mathbf{x})]
$$
まず、$\ln p(\mathbf{x})$ を展開する。
$$
\ln p(\mathbf{x}) = \ln \left[ \frac{1}{(2\pi)^{D/2}|\boldsymbol{\Sigma}|^{1/2}} \right] - \frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})
$$
$$
= -\frac{D}{2}\ln(2\pi) - \frac{1}{2}\ln|\boldsymbol{\Sigma}| - \frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})
$$
これをエントロピーの式に代入して期待値をとる。第1項と第2項は定数であるから、
$$
H[\mathbf{x}] = \frac{D}{2}\ln(2\pi) + \frac{1}{2}\ln|\boldsymbol{\Sigma}| + \frac{1}{2}\mathbb{E} \left[ (\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) \right]
$$
次に、期待値の部分 $\mathbb{E} \left[ (\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) \right]$ を計算する。
行列のトレース（跡）の性質 $\text{Tr}(AB) = \text{Tr}(BA)$、およびスカラー量はそれ自身のトレースと等しいこと（$a = \text{Tr}(a)$）を利用する。
$$
(\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) = \text{Tr} \left( (\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu}) \right) = \text{Tr} \left( \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})(\mathbf{x}-\boldsymbol{\mu})^T \right)
$$
期待値とトレースの順序を交換すると、
$$
\mathbb{E} \left[ \text{Tr} \left( \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})(\mathbf{x}-\boldsymbol{\mu})^T \right) \right] = \text{Tr} \left( \boldsymbol{\Sigma}^{-1} \mathbb{E} \left[ (\mathbf{x}-\boldsymbol{\mu})(\mathbf{x}-\boldsymbol{\mu})^T \right] \right)
$$
ここで、共分散行列の定義より $\mathbb{E} \left[ (\mathbf{x}-\boldsymbol{\mu})(\mathbf{x}-\boldsymbol{\mu})^T \right] = \boldsymbol{\Sigma}$ である。
よって、
$$
\text{Tr} \left( \boldsymbol{\Sigma}^{-1} \boldsymbol{\Sigma} \right) = \text{Tr}(\mathbf{I}_D) = D
$$
となる（$\mathbf{I}_D$ は $D \times D$ の単位行列）。
したがって、期待値の項は $\frac{D}{2}$ となる。

最終的にこれをエントロピーの式に戻すと、
$$
H[\mathbf{x}] = \frac{D}{2}\ln(2\pi) + \frac{1}{2}\ln|\boldsymbol{\Sigma}| + \frac{D}{2}
$$
$$
= \frac{1}{2}\ln|\boldsymbol{\Sigma}| + \frac{D}{2}(1 + \ln(2\pi))
$$
となり、題意の式が証明された。


## Exercise 1.36 ⭐

## Exercise 1.36: 凸関数と2階微分 (Convex Functions and Second Derivatives)

### 問題
狭義の凸関数 (strictly convex function) は、関数上の任意の2点を結ぶ弦 (chord) が、常にその関数のグラフより上側に位置するような関数として定義される。この条件が、関数の2階微分が常に正であることと等価であることを示せ。

### 解答と解説

1変数関数 $f(x)$ が2回連続微分可能であると仮定します。

**1. 弦が関数の上にある $\implies$ 2階微分が正**

凸関数の定義から、任意の $a, b$ (ただし $a \neq b$) と $0 < \lambda < 1$ に対して次が成り立ちます：
$$ f(\lambda a + (1 - \lambda)b) < \lambda f(a) + (1 - \lambda)f(b) $$

ここで、$b = x$, $a = x + \Delta x$ と置きます。
$$ f(x + \lambda \Delta x) < \lambda f(x + \Delta x) + (1 - \lambda)f(x) $$

関数 $f(x)$ を $x$ の周りで2次までテイラー展開します：
$$ f(x + \epsilon) = f(x) + \epsilon f'(x) + \frac{\epsilon^2}{2} f''(x) + O(\epsilon^3) $$

左辺 $f(x + \lambda \Delta x)$ に適用すると：
$$ f(x + \lambda \Delta x) = f(x) + \lambda \Delta x f'(x) + \frac{\lambda^2 (\Delta x)^2}{2} f''(x) + O((\Delta x)^3) $$

右辺の $f(x + \Delta x)$ に適用すると：
$$ \lambda f(x + \Delta x) + (1 - \lambda)f(x) = \lambda \left( f(x) + \Delta x f'(x) + \frac{(\Delta x)^2}{2} f''(x) + O((\Delta x)^3) \right) + (1 - \lambda)f(x) $$
$$ = f(x) + \lambda \Delta x f'(x) + \frac{\lambda (\Delta x)^2}{2} f''(x) + O((\Delta x)^3) $$

これを元の不等式に代入して整理すると：
$$ \frac{\lambda^2 (\Delta x)^2}{2} f''(x) < \frac{\lambda (\Delta x)^2}{2} f''(x) $$
$$ \lambda^2 f''(x) < \lambda f''(x) $$

$0 < \lambda < 1$ より $\lambda - \lambda^2 > 0$ であるため、
$$ \lambda(1 - \lambda) f''(x) > 0 \implies f''(x) > 0 $$
となり、2階微分が正であることが導かれます。

**2. 2階微分が正 $\implies$ 弦が関数の上にある**

逆に $f''(x) > 0$ と仮定します。平均値の定理の拡張（テイラーの定理の剰余項）を利用します。
$z = \lambda a + (1 - \lambda)b$ とします。
$f(a)$ と $f(b)$ を $z$ の周りで展開すると：
$$ f(a) = f(z) + (a - z)f'(z) + \frac{(a - z)^2}{2}f''(c_1) \quad (c_1 \text{ は } a, z \text{ の間}) $$
$$ f(b) = f(z) + (b - z)f'(z) + \frac{(b - z)^2}{2}f''(c_2) \quad (c_2 \text{ は } b, z \text{ の間}) $$

これらの式にそれぞれ $\lambda$ と $(1 - \lambda)$ を掛けて足し合わせます：
$$ \lambda f(a) + (1 - \lambda)f(b) = f(z) + (\lambda a + (1 - \lambda)b - z)f'(z) + \lambda \frac{(a - z)^2}{2}f''(c_1) + (1 - \lambda)\frac{(b - z)^2}{2}f''(c_2) $$

ここで $z = \lambda a + (1 - \lambda)b$ より、1次微分の項はゼロになります。
また、$f''(x) > 0$ より、剰余項は両方とも正になります。したがって、
$$ \lambda f(a) + (1 - \lambda)f(b) > f(z) = f(\lambda a + (1 - \lambda)b) $$
となり、狭義の凸関数の定義を満たします。

以上より、2つの条件は等価であることが証明されました。


## Exercise 1.37

## Exercise 1.37: 条件付きエントロピーと同時エントロピー (Conditional and Joint Entropy)

### 問題
条件付きエントロピーの定義 (1.111) と確率の乗法定理 (product rule) を用いて、式 (1.112) の関係 $H[x, y] = H[y|x] + H[x]$ を証明せよ。

### 解答と解説

条件付きエントロピー $H[y|x]$ は、テキストの式 (1.111) に従って以下のように定義されます。
$$ H[y|x] = - \iint p(y, x) \ln p(y|x) \, dy \, dx $$

また、確率の乗法定理により、同時確率 $p(x, y)$ は条件付き確率 $p(y|x)$ と周辺確率 $p(x)$ の積として表すことができます。
$$ p(x, y) = p(y|x) p(x) $$
これを変形すると、
$$ p(y|x) = \frac{p(x, y)}{p(x)} = \frac{p(y, x)}{p(x)} $$
となります（$p(x, y) = p(y, x)$ であるため）。

この式を条件付きエントロピーの定義式に代入します。
$$ H[y|x] = - \iint p(y, x) \ln \left( \frac{p(y, x)}{p(x)} \right) \, dy \, dx $$

対数の性質 $\ln(A / B) = \ln A - \ln B$ を用いて展開します。
$$ H[y|x] = - \iint p(y, x) \left( \ln p(y, x) - \ln p(x) \right) \, dy \, dx $$
$$ = - \iint p(y, x) \ln p(y, x) \, dy \, dx + \iint p(y, x) \ln p(x) \, dy \, dx $$

ここで、第1項は同時確率分布 $p(y, x)$ に対する微分エントロピーの定義そのものであり、$H[x, y]$ となります。
$$ - \iint p(y, x) \ln p(y, x) \, dy \, dx = H[x, y] $$

一方、第2項については、$x$ に依存しない積分 $\int p(y, x) \, dy$ を先に計算します（確率の加法定理）。周辺確率の定義 $\int p(y, x) \, dy = p(x)$ を用いると、
$$ \iint p(y, x) \ln p(x) \, dy \, dx = \int \left( \int p(y, x) \, dy \right) \ln p(x) \, dx $$
$$ = \int p(x) \ln p(x) \, dx $$
となります。これは周辺確率分布 $p(x)$ の微分エントロピーの符号を反転させたもの、すなわち $-H[x]$ です。

したがって、これらをまとめると、
$$ H[y|x] = H[x, y] - H[x] $$
が得られます。この式を移項することで、
$$ H[x, y] = H[y|x] + H[x] $$
となり、式 (1.112) が証明されました。

**物理的・情報論的解釈**：
この等式は、確率変数 $x$ と $y$ の両方を記述するために必要な総情報量 $H[x, y]$ が、「$x$ を単独で記述するために必要な情報量 $H[x]$」と、「$x$ が既に知られているという条件のもとで $y$ を追加で記述するために必要な情報量 $H[y|x]$」の和に等しいことを意味しています。


## Exercise 1.38

## Exercise 1.38: イェンセンの不等式と数学的帰納法 (Jensen's Inequality and Mathematical Induction)

### 問題
数学的帰納法を用いて、凸関数の不等式 (1.114) がイェンセンの不等式 (1.115) を意味することを示せ。

### 解答と解説

関数 $f(x)$ を凸関数とします。凸関数の定義（式1.114）から、$0 \leq \lambda \leq 1$ に対して次の不等式が成り立ちます：
$$ f(\lambda a + (1 - \lambda)b) \leq \lambda f(a) + (1 - \lambda)f(b) \quad \text{--- (A)} $$

証明すべきイェンセンの不等式（式1.115）は、非負で和が1となる係数 $\lambda_1, \ldots, \lambda_M$ （すなわち $\lambda_i \geq 0$, $\sum_{i=1}^M \lambda_i = 1$）に対して次が成り立つというものです：
$$ f\left( \sum_{i=1}^M \lambda_i x_i \right) \leq \sum_{i=1}^M \lambda_i f(x_i) \quad \text{--- (B)} $$

この命題 (B) を、変数の数 $M$ に対する数学的帰納法を用いて証明します。

**1. $M=1$ および $M=2$ の場合**

- $M=1$ のとき、条件 $\sum_{i=1}^1 \lambda_i = 1$ より $\lambda_1 = 1$ となります。
  $$ f(\lambda_1 x_1) = f(x_1) \leq 1 \cdot f(x_1) $$
  となり、明らかに成立します。
- $M=2$ のとき、$\lambda_1 + \lambda_2 = 1$ より $\lambda_2 = 1 - \lambda_1$ です。したがって、
  $$ f(\lambda_1 x_1 + \lambda_2 x_2) = f(\lambda_1 x_1 + (1 - \lambda_1)x_2) \leq \lambda_1 f(x_1) + (1 - \lambda_1)f(x_2) = \lambda_1 f(x_1) + \lambda_2 f(x_2) $$
  これは凸関数の定義 (A) そのものであり、成立します。

**2. $M=k$ のときの仮定**

ある整数 $k \geq 2$ に対して、命題 (B) が成り立つと仮定します。すなわち、$\sum_{i=1}^k \eta_i = 1$ かつ $\eta_i \geq 0$ を満たす任意の $\{\eta_i\}$ に対して、次が成り立つとします：
$$ f\left( \sum_{i=1}^k \eta_i x_i \right) \leq \sum_{i=1}^k \eta_i f(x_i) \quad \text{--- (C)} $$

**3. $M=k+1$ の場合の証明**

$M = k+1$ の場合を考えます。和が1となる係数 $\lambda_1, \ldots, \lambda_{k+1}$ に対する関数の値は、次のように書けます：
$$ f\left( \sum_{i=1}^{k+1} \lambda_i x_i \right) = f\left( \sum_{i=1}^k \lambda_i x_i + \lambda_{k+1} x_{k+1} \right) $$

ここで、$S_k = \sum_{i=1}^k \lambda_i$ と置きます。$\sum_{i=1}^{k+1} \lambda_i = 1$ より、$S_k + \lambda_{k+1} = 1$ です。
もし $S_k = 0$ ならば、$\lambda_1 = \dots = \lambda_k = 0, \lambda_{k+1} = 1$ となり自明に成立するため、$S_k > 0$ と仮定して進めます。

総和を次のように変形します：
$$ \sum_{i=1}^k \lambda_i x_i + \lambda_{k+1} x_{k+1} = S_k \left( \sum_{i=1}^k \frac{\lambda_i}{S_k} x_i \right) + \lambda_{k+1} x_{k+1} $$

$S_k + \lambda_{k+1} = 1$ であるため、$M=2$ の場合の不等式（凸関数の定義）を適用することができます：
$$ f\left( S_k \left( \sum_{i=1}^k \frac{\lambda_i}{S_k} x_i \right) + \lambda_{k+1} x_{k+1} \right) \leq S_k f\left( \sum_{i=1}^k \frac{\lambda_i}{S_k} x_i \right) + \lambda_{k+1} f(x_{k+1}) $$

次に、右辺の第1項に注目します。係数 $\eta_i = \frac{\lambda_i}{S_k}$ を考えると、$\sum_{i=1}^k \eta_i = \sum_{i=1}^k \frac{\lambda_i}{S_k} = \frac{S_k}{S_k} = 1$ を満たします。
したがって、帰納法の仮定 (C) を適用できます：
$$ f\left( \sum_{i=1}^k \frac{\lambda_i}{S_k} x_i \right) \leq \sum_{i=1}^k \frac{\lambda_i}{S_k} f(x_i) $$

これを先ほどの式に代入します：
$$ S_k f\left( \sum_{i=1}^k \frac{\lambda_i}{S_k} x_i \right) + \lambda_{k+1} f(x_{k+1}) \leq S_k \left( \sum_{i=1}^k \frac{\lambda_i}{S_k} f(x_i) \right) + \lambda_{k+1} f(x_{k+1}) $$
$$ = \sum_{i=1}^k \lambda_i f(x_i) + \lambda_{k+1} f(x_{k+1}) = \sum_{i=1}^{k+1} \lambda_i f(x_i) $$

以上の結果をまとめると、
$$ f\left( \sum_{i=1}^{k+1} \lambda_i x_i \right) \leq \sum_{i=1}^{k+1} \lambda_i f(x_i) $$
となり、$M=k+1$ の場合でも命題 (B) が成立することが示されました。

したがって、数学的帰納法により、すべての自然数 $M$ に対してイェンセンの不等式が成り立つことが証明されました。


## Exercise 1.39

## Exercise 1.39: 同時分布とエントロピー (Joint Distribution and Entropy)

### 問題
表1.3で与えられる2つの2値確率変数 $x, y$ の同時分布を考える。
以下の各量を計算せよ。（対数の底は自然対数 $e$ を用いて計算するが、底を2としてビット(bit)単位で計算しても良い）
(a) $H[x]$ (b) $H[y]$ (c) $H[y|x]$ (d) $H[x|y]$ (e) $H[x, y]$ (f) $I[x, y]$
また、これらの量の関係を示す図を描け（本解答では文章と関係式で説明する）。

**表1.3の再掲**
| $y \setminus x$ | $x = 0$ | $x = 1$ | 周辺確率 $p(y)$ |
| :---: | :---: | :---: | :---: |
| **$y = 0$** | $1/3$ | $0$ | $1/3$ |
| **$y = 1$** | $1/3$ | $1/3$ | $2/3$ |
| **周辺確率 $p(x)$**| $2/3$ | $1/3$ | $1$ |

### 解答と解説

まず、表から周辺確率分布を求めます。
- $p(x=0) = 1/3 + 1/3 = 2/3$
- $p(x=1) = 0 + 1/3 = 1/3$
- $p(y=0) = 1/3 + 0 = 1/3$
- $p(y=1) = 1/3 + 1/3 = 2/3$

**解 (a): $H[x]$ の計算**
周辺確率 $p(x)$ のエントロピーを計算します。
$$ H[x] = - \sum_x p(x) \ln p(x) = - \left( \frac{2}{3} \ln \frac{2}{3} + \frac{1}{3} \ln \frac{1}{3} \right) $$
$$ = - \left( \frac{2}{3} (\ln 2 - \ln 3) + \frac{1}{3} (0 - \ln 3) \right) = \ln 3 - \frac{2}{3} \ln 2 $$

**解 (b): $H[y]$ の計算**
周辺確率 $p(y)$ のエントロピーを計算します。$p(y)$ は $p(x)$ と確率の組が同じなので、値も等しくなります。
$$ H[y] = - \sum_y p(y) \ln p(y) = - \left( \frac{1}{3} \ln \frac{1}{3} + \frac{2}{3} \ln \frac{2}{3} \right) = \ln 3 - \frac{2}{3} \ln 2 $$

**解 (e): $H[x, y]$ の計算**
同時分布のエントロピーは、ゼロでない確率要素のみで計算します。
$$ H[x, y] = - \sum_{x} \sum_{y} p(x, y) \ln p(x, y) = - 3 \times \left( \frac{1}{3} \ln \frac{1}{3} \right) = - \ln \frac{1}{3} = \ln 3 $$

**解 (c): $H[y|x]$ の計算**
条件付きエントロピーは $H[y|x] = H[x, y] - H[x]$ を用いて簡単に求められます。
$$ H[y|x] = \ln 3 - \left( \ln 3 - \frac{2}{3} \ln 2 \right) = \frac{2}{3} \ln 2 $$

（※ 直接計算する場合は以下のようになります）
$p(y|x=0) = \{1/2, 1/2\}$ より $H[y|x=0] = \ln 2$。
$p(y|x=1) = \{0, 1\}$ より $H[y|x=1] = 0$。
$H[y|x] = p(x=0)H[y|x=0] + p(x=1)H[y|x=1] = \frac{2}{3} \ln 2 + \frac{1}{3} \times 0 = \frac{2}{3} \ln 2$

**解 (d): $H[x|y]$ の計算**
同様に $H[x|y] = H[x, y] - H[y]$ を用います。
$$ H[x|y] = \ln 3 - \left( \ln 3 - \frac{2}{3} \ln 2 \right) = \frac{2}{3} \ln 2 $$

**解 (f): $I[x, y]$ の計算**
相互情報量 $I[x, y]$ は $I[x, y] = H[x] - H[x|y]$ 等の公式から求められます。
$$ I[x, y] = H[x] - H[x|y] = \left( \ln 3 - \frac{2}{3} \ln 2 \right) - \frac{2}{3} \ln 2 = \ln 3 - \frac{4}{3} \ln 2 $$

### ベン図による関係図の記述

エントロピーの関係は、2つの円が重なるベン図（情報ダイアグラム）で視覚化できます。

```mermaid
venn
    title "Entropy and Mutual Information Relationship"
    A(H[x]): 1
    B(H[y]): 1
    Intersection(I[x,y]): 0.4
```
※ 面積の大まかな内訳は以下の通りです：
* **全体 (和集合)**: $H[x, y] = \ln 3 \approx 1.098$
* **左側の円**: $H[x] = \ln 3 - \frac{2}{3}\ln 2 \approx 0.636$
* **右側の円**: $H[y] = \ln 3 - \frac{2}{3}\ln 2 \approx 0.636$
* **重なり部分**: 相互情報量 $I[x,y] = \ln 3 - \frac{4}{3}\ln 2 \approx 0.174$
* **左側の円の単独部分**: $H[x|y] = \frac{2}{3}\ln 2 \approx 0.462$
* **右側の円の単独部分**: $H[y|x] = \frac{2}{3}\ln 2 \approx 0.462$

このように、$H[x, y] = H[x|y] + I[x, y] + H[y|x]$ という関係が視覚的にも確認できます。


## Exercise 1.40

## Exercise 1.40: 相加平均と相乗平均の関係 (AM-GM Inequality)

### 問題
関数 $f(x) = \ln x$ にイェンセンの不等式 (1.115) を適用することにより、実数の集合の相加平均（算術平均）が、相乗平均（幾何平均）を下回ることはないことを示せ。

### 解答と解説

$N$ 個の正の実数 $x_1, x_2, \ldots, x_N$ を考えます。これらに対する相加平均 (Arithmetic Mean) と相乗平均 (Geometric Mean) は次のように定義されます：
- 相加平均： $A = \frac{1}{N} \sum_{i=1}^N x_i$
- 相乗平均： $G = \left( \prod_{i=1}^N x_i \right)^{1/N}$

証明すべき不等式は $A \geq G$ です。

まず、問題文で指定された関数 $f(x) = \ln x$ の性質を確認します。
この関数の2階微分は $f''(x) = -1/x^2 < 0$ となり、常に負です。これは $f(x) = \ln x$ が **上に凸な関数 (concave function)** であることを意味します。

イェンセンの不等式 (1.115) は通常、下に凸な関数 (convex function) $g(x)$ に対して $g(E[x]) \leq E[g(x)]$ として定義されます。
ここでは $g(x) = - \ln x$ と置くと、$g''(x) = 1/x^2 > 0$ となり下に凸な関数になるため、イェンセンの不等式を適用できます。
等確率の係数として $\lambda_i = \frac{1}{N}$ （ここで $\sum_{i=1}^N \lambda_i = 1$）を設定します。

イェンセンの不等式より：
$$ g\left( \sum_{i=1}^N \lambda_i x_i \right) \leq \sum_{i=1}^N \lambda_i g(x_i) $$

これに $g(x) = -\ln x$ と $\lambda_i = \frac{1}{N}$ を代入します：
$$ - \ln \left( \frac{1}{N} \sum_{i=1}^N x_i \right) \leq \sum_{i=1}^N \frac{1}{N} (-\ln x_i) $$

両辺に $-1$ を掛けます。このとき不等号の向きが反転することに注意します（これは上に凸な関数 $\ln x$ に直接イェンセンの不等式を適用した形になります）：
$$ \ln \left( \frac{1}{N} \sum_{i=1}^N x_i \right) \geq \frac{1}{N} \sum_{i=1}^N \ln x_i $$

右辺の対数の和を整理します。対数の性質 $a \ln b = \ln(b^a)$ と $\sum \ln b_i = \ln (\prod b_i)$ を用います：
$$ \frac{1}{N} \sum_{i=1}^N \ln x_i = \sum_{i=1}^N \ln (x_i^{1/N}) = \ln \left( \prod_{i=1}^N x_i^{1/N} \right) = \ln \left( \left( \prod_{i=1}^N x_i \right)^{1/N} \right) $$

したがって、不等式は次のようになります：
$$ \ln \left( \frac{1}{N} \sum_{i=1}^N x_i \right) \geq \ln \left( \left( \prod_{i=1}^N x_i \right)^{1/N} \right) $$

自然対数関数 $\ln(x)$ は単調増加関数（$x_1 > x_2 \iff \ln x_1 > \ln x_2$）であるため、対数を外しても不等号の関係はそのまま保たれます：
$$ \frac{1}{N} \sum_{i=1}^N x_i \geq \left( \prod_{i=1}^N x_i \right)^{1/N} $$

左辺は相加平均、右辺は相乗平均を表しており、これで相加平均が相乗平均を下回らないこと（相加相乗平均の不等式）が証明されました。


## Exercise 1.41 ⭐

## Exercise 1.41: 相互情報量とエントロピーの関係 (Mutual Information and Entropy)

### 問題
確率の加法定理 (sum rule) と乗法定理 (product rule) を用いて、相互情報量 $I[x, y]$ が関係式 (1.121) を満たすことを示せ。
$$ I[x, y] = H[x] - H[x|y] = H[y] - H[y|x] = H[x] + H[y] - H[x, y] $$

### 解答と解説

相互情報量 $I[x, y]$ の定義式から出発します。
$$ I[x, y] = \iint p(x, y) \ln \frac{p(x, y)}{p(x)p(y)} \, dx \, dy $$

対数の性質 $\ln(A / B) = \ln A - \ln B$ および $\ln(AB) = \ln A + \ln B$ を用いて展開します。
$$ \ln \frac{p(x, y)}{p(x)p(y)} = \ln p(x, y) - \ln p(x) - \ln p(y) $$

これを相互情報量の積分に代入し、3つの項に分割します：
$$ I[x, y] = \iint p(x, y) \ln p(x, y) \, dx \, dy - \iint p(x, y) \ln p(x) \, dx \, dy - \iint p(x, y) \ln p(y) \, dx \, dy $$

各項をエントロピーの定義に当てはめて計算していきます。

**第1項:**
これは同時エントロピー $H[x, y]$ の定義の符号を反転させたものです。
$$ \iint p(x, y) \ln p(x, y) \, dx \, dy = -H[x, y] $$

**第2項:**
積分順序を交換し、確率の加法定理（周辺化：$\int p(x, y) \, dy = p(x)$）を用います。
$$ - \iint p(x, y) \ln p(x) \, dx \, dy = - \int \left( \int p(x, y) \, dy \right) \ln p(x) \, dx $$
$$ = - \int p(x) \ln p(x) \, dx = H[x] $$

**第3項:**
同様に、確率の加法定理（$\int p(x, y) \, dx = p(y)$）を用います。
$$ - \iint p(x, y) \ln p(y) \, dx \, dy = - \int \left( \int p(x, y) \, dx \right) \ln p(y) \, dy $$
$$ = - \int p(y) \ln p(y) \, dy = H[y] $$

これらを足し合わせると、関係式の3つ目の形が得られます。
$$ I[x, y] = -H[x, y] + H[x] + H[y] = H[x] + H[y] - H[x, y] \quad \text{--- (1)} $$

さらに、確率の乗法定理 $p(x, y) = p(y|x)p(x)$ を用いて導出された式 (1.112) の関係 $H[x, y] = H[y|x] + H[x]$ を式(1)に代入します。
$$ I[x, y] = H[x] + H[y] - (H[y|x] + H[x]) $$
$$ = H[y] - H[y|x] \quad \text{--- (2)} $$

同様に、$x$ と $y$ の対称性から $p(x, y) = p(x|y)p(y)$ より得られる関係 $H[x, y] = H[x|y] + H[y]$ を式(1)に代入します。
$$ I[x, y] = H[x] + H[y] - (H[x|y] + H[y]) $$
$$ = H[x] - H[x|y] \quad \text{--- (3)} $$

以上、(1), (2), (3) により、求める関係式がすべて示されました。
$$ I[x, y] = H[x] - H[x|y] = H[y] - H[y|x] = H[x] + H[y] - H[x, y] $$

**物理的・情報論的解釈**：
相互情報量 $I[x, y]$ は、$y$ を知ることで $x$ の不確実性がどれだけ減少するか（またはその逆）を表します。式 $H[x] - H[x|y]$ は、「$x$ が元々持っていた不確実性」から「$y$ を知った後でも残る $x$ の不確実性」を引いたものであり、これが $y$ によって得られた $x$ に関する情報量に他ならないことを数式で示しています。


<!-- ORIGINAL GROUPED TEXT:

# 演習問題 1.28

## 問題文
事象の確率 $p$ のみに依存する情報量（自己情報量）を $h(p)$ とする。独立な2つの事象が同時に起こる場合の情報量が、それぞれの情報量の和になること（加法性）を仮定する。すなわち、
$$ h(p_1 p_2) = h(p_1) + h(p_2) $$
が成り立つとする。この関数方程式を満たす連続関数 $h(p)$ が、定数倍を除いて対数関数 $\ln p$ になること（$h(p) \propto \ln p$）を示せ。

## 解答と解説
### 1. 関数方程式の性質（整数および有理数の場合）
与えられた関数方程式は以下の形をしています。
$$ h(xy) = h(x) + h(y) \quad (0 < x, y \le 1) $$
まず、任意の正の整数 $n$ に対して、帰納法により次が成り立つことが示せます。
$$ h(x^n) = n h(x) $$
（証明：$n=1$ は自明。$n=2$ のとき $h(x^2) = h(xx) = h(x)+h(x) = 2h(x)$。$n=k$ で成立すると仮定すると、$h(x^{k+1}) = h(x^k x) = h(x^k) + h(x) = k h(x) + h(x) = (k+1)h(x)$ となり成立。）

次に、任意の正の有理数 $r = \frac{m}{n}$ （$m, n$ は正の整数）についても調べます。
$y = x^{\frac{m}{n}}$ と置きます。両辺を $n$ 乗すると $y^n = x^m$ です。
関数方程式より、
$$ h(y^n) = h(x^m) \implies n h(y) = m h(x) \implies h(y) = \frac{m}{n} h(x) $$
したがって、有理数 $r$ においても $h(x^r) = r h(x)$ が成り立ちます。

### 2. 連続性による実数への拡張
問題の仮定（または物理的・情報的要請）として $h(p)$ は確率 $p$ に関して連続な関数です。
有理数は実数の中に稠密に存在するため、任意の正の実数 $a$ に対して $r_k \to a$ となる有理数列 $\{r_k\}$ をとることができます。
連続関数であれば極限と関数の適用を交換できるため、任意の実数 $a$ に対して、
$$ h(x^a) = h(\lim_{k \to \infty} x^{r_k}) = \lim_{k \to \infty} h(x^{r_k}) = \lim_{k \to \infty} r_k h(x) = a h(x) $$
が成り立ちます。

### 3. 対数関数との関係の導出
関数が $h(x^a) = a h(x)$ を満たすことがわかりました。
ここで、底をネイピア数 $e$ とした指数表現を考えます。任意の確率 $p \ (0 < p \le 1)$ を $p = e^{\ln p}$ と書くことができます。
これを先ほどの実数乗の公式に適用します。$x = e$, $a = \ln p$ と置くと、
$$ h(p) = h(e^{\ln p}) = (\ln p) \cdot h(e) $$
ここで、$h(e)$ は変数 $p$ に依存しない単なる定数です。これを $C = h(e)$ と置くと、
$$ h(p) = C \ln p $$
と表せます。情報量は常に非負であり、確率が小さいほど情報量は大きくなるべきなので、$h(p) \ge 0$ を満たすために定数 $C$ は負の定数となります（通常 $C = -1$ などとします）。

これにより、関数 $h(p)$ は $\ln p$ に比例すること、すなわち $h(p) \propto \ln p$ が証明されました。


In [11]:
# Exercise 1.39 数値検証: 2変量正規分布の相互情報量 I(x, y) = -0.5 * ln(1 - rho^2)
rhos = [0.0, 0.3, 0.6, 0.9, 0.99]
print("Exercise 1.39 相関係数 rho と相互情報量 I(x, y) (nats):")
for rho in rhos:
    # 共分散行列 Sigma = [[1, rho], [rho, 1]]
    # det(Sigma) = 1 - rho^2
    I_analytical = -0.5 * np.log(1.0 - rho**2)
    print(f"  rho = {rho:4.2f}: I(x, y) = {I_analytical:.4f} nats")
print("  => 相関 |rho| -> 1 で相互情報量は無限大に発散、rho=0 で I=0 (独立) を確認 (OK)")

Exercise 1.39 相関係数 rho と相互情報量 I(x, y) (nats):
  rho = 0.00: I(x, y) = -0.0000 nats
  rho = 0.30: I(x, y) = 0.0472 nats
  rho = 0.60: I(x, y) = 0.2231 nats
  rho = 0.90: I(x, y) = 0.8304 nats
  rho = 0.99: I(x, y) = 1.9585 nats
  => 相関 |rho| -> 1 で相互情報量は無限大に発散、rho=0 で I=0 (独立) を確認 (OK)


## 第1章 演習問題 総括

本ノートブックでは、PRML 第1章の全41問（Exercises 1.1 〜 1.41）にわたる重要テーマ：
1. **多項式モデルと基底関数展開における正規方程式の厳密な導出**
2. **確率密度の変数変換定理と最頻値の非線形シフトの数学的本質**
3. **最尤分散推定量の過小バイアス ($N-1$ 自由度への補正)**
4. **高次元幾何学（次元の呪い、球殻体積集中、ランダムベクトルの直交性）**
5. **決定理論における条件付き平均（L2損失）と条件付き中央値（L1損失）**
6. **情報理論の金字塔：ガウス分布の最大エントロピー定理、KLダイバージェンスの非負性、相互情報量の幾何学**

を数理証明と Python による直接数値シミュレーションによって完全に解明・検証しました。